# Literature topics and Y/N triage decisions

Classify papers from **Full.xlsx** and compare them with papers whose selected decision
column is exactly **Y** in **SELECTED 7000.xlsx**.
The main chart compares **Y (green)** with **N (orange)**, using explicit decisions
from **Full_gpt-5.xlsx / Agent_YN**. Tables show **Before triage**, **After triage (Y)**,
and **After triage (N)** together, with unresolved decisions reported separately.

The four displayed categories are **Chemical synthesis**, **Theory & modeling**,
**Crystal engineering**, and **Functional materials**. Classification uses each paper's
**title and abstract**, plus document-type metadata, not its journal or triage decision.

**Synthesis-inclusive definition:** Chemical synthesis includes the previous synthetic
methods/transformations topic **plus original papers that explicitly report experimental
MOF or coordination-framework preparation**, even when structure or applications are
their main topic. The original primary topic is preserved separately in the audit.
This is a documented change in category scope, not a change to the saved Y/N decisions.

**Review rule: every identified review article is assigned to Theory & modeling**,
regardless of whether it reviews synthesis, crystal engineering, or applications.
This is a user-defined category convention; it does not imply the review conducted
computational research. Reviews are detected from Document Type or explicit
self-descriptions in the title/abstract. Generic mentions of a perspective or recent
advances alone do not trigger this rule. Detection evidence is exported for inspection.

**Interpretation:** these are rough, deterministic topic estimates, **not classifications
generated by an LLM** and not validated expert labels. Overlapping topics receive one
suggested primary category and a review flag. Broader fallback rules classify papers
missed by specific phrases and expose their weak evidence for review. Missing text
remains an explicit exception. The counts include flagged suggestions.
The original main-contribution rules still produce `primary_topic_category`. The
displayed `category` applies the synthesis-inclusive rule afterward. A reported
preparation statement must concern a framework, rather than a molecular product made
with a MOF catalyst, a MOF-derived material alone, or a hypothetical/background example.
The classifier receives title, abstract, and document type only; desired Y/N proportions are never inputs.

**Topic and eligibility are different.** The project's current
[triage rubric](../MOFinder/prompts/abstract_triage.txt) targets traditional solution-phase
MOF synthesis and excludes some real synthesis work, including mechanochemistry,
film-only growth, and post-synthetic-modification-only studies. A synthesis paper can
therefore be N. This notebook uses stored decisions; the current prompt provides context
and does not establish which prompt version produced the historical decisions.

Run all cells in order. This notebook is self-contained: its analysis code is embedded,
and no API key, network service, or companion Python module is needed.

## 1. Input files and settings

The default inputs are three original workbooks in the MOF project's `14489 paper`
folder. Change `DATA_DIR` if you move them, or put all three beside the notebook.
`gpt-5` is the default triage column; other available decision columns are also summarized.
The notebook only reads the original workbooks. Outputs go into a separate folder.

If packages are missing, uncomment the installation cell below and run it once.

In [ ]:
# %pip install pandas openpyxl matplotlib nbformat nbclient ipykernel

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import platform
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

INPUT_NAMES = ['Full.xlsx', 'SELECTED 7000.xlsx', 'Full_gpt-5.xlsx']
INPUT_CANDIDATES = [Path.cwd(), Path.cwd().parent / '14489 paper', Path.cwd() / '14489 paper']
DATA_DIR = next((p for p in INPUT_CANDIDATES if all((p / name).is_file() for name in INPUT_NAMES)),
                Path.cwd().parent / '14489 paper')

BEFORE_PATH = DATA_DIR / 'Full.xlsx'
AFTER_PATH = DATA_DIR / 'SELECTED 7000.xlsx'
DECISION_PATH = DATA_DIR / 'Full_gpt-5.xlsx'
DECISION_COLUMN = 'Agent_YN'
SHEET_NAME = 'Sheet1'
LABEL_COLUMN = 'gpt-5'  # Alternatives: 'gpt-4o-mini', 'ground truth by gpt-5'
# If changing the triage model, also set the corresponding full decision workbook.
PRIOR_TOPICS_PATH = Path.cwd() / 'baseline_v1_topics.csv'
PREVIOUS_TOPICS_PATH = Path.cwd() / 'archive' / 'v3' / 'paper_topics.csv'
PRIMARY_TOPICS_PATH = Path.cwd() / 'archive' / 'v4' / 'paper_topics.csv'

OUTPUT_DIR = Path.cwd() / 'outputs'
OVERRIDE_PATH = Path.cwd() / 'category_overrides.csv'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.max_columns', 12)
print('Before:', BEFORE_PATH)
print('After: ', AFTER_PATH)
print('Explicit Y/N:', DECISION_PATH, '/', DECISION_COLUMN)
print('After-triage rule:', LABEL_COLUMN, '== Y (case-insensitive; whitespace trimmed)')
print('Output:', OUTPUT_DIR)

## 2. Read records, match papers, and check labels

**Counting unit:** the main table counts unique papers by normalized DOI, with WOS ID
and then normalized title as fallbacks when DOI is absent. The row-count table preserves
the original spreadsheet counts. Duplicate DOI records are retained in row exports;
their longest available title/abstract record represents the paper for classification.

Selected papers match the before set by DOI, then WOS ID, then an unambiguous title.
Conflicting identifiers are reported rather than silently merged. For duplicate records,
a paper is retained if **any** of its rows has the requested Y label; conflicting decisions
are reported. No download flag, numeric label, citation flag, or blank cell is treated as Y.

In [ ]:
"""Local, auditable identity matching and counts for literature triage."""
import html
import re
import unicodedata
from pathlib import Path

import pandas as pd


TRIAGE_COLUMNS = ['gpt-5', 'gpt-4o-mini', 'ground truth by gpt-5', 'Agent_YN']
METADATA_COLUMNS = ['Article Title', 'Abstract', 'DOI', 'UT (Unique WOS ID)',
                    'Publication Year', 'Source Title', 'Document Type']


def normalize_text(value):
    if value is None or pd.isna(value):
        return ''
    return unicodedata.normalize('NFKC', html.unescape(str(value))).strip()


def normalize_doi(value):
    value = normalize_text(value).casefold()
    value = re.sub(r'^https?://(?:dx\.)?doi\.org/', '', value)
    value = re.sub(r'^doi\s*:\s*', '', value)
    return value.strip()


def normalize_title(value):
    return ''.join(c for c in normalize_text(value).casefold() if c.isalnum())


def is_y(values):
    """Only an exact Y, after trimming and case normalization, is included."""
    return values.fillna('').astype(str).str.strip().str.upper().eq('Y')


def prepare_records(frame, source):
    frame = frame.copy().fillna('')
    for col in METADATA_COLUMNS:
        if col not in frame:
            frame[col] = ''
    frame['source_file'] = str(source)
    frame['excel_row'] = range(2, len(frame) + 2)
    frame['doi_key'] = frame['DOI'].map(normalize_doi)
    frame['wos_key'] = frame['UT (Unique WOS ID)'].map(lambda x: normalize_text(x).casefold())
    frame['title_key'] = frame['Article Title'].map(normalize_title)
    frame['paper_id'] = [
        'doi:' + doi if doi else 'wos:' + wos if wos else 'title:' + title if title
        else 'row:{}:{}'.format(Path(source).name, row)
        for doi, wos, title, row in zip(frame.doi_key, frame.wos_key,
                                      frame.title_key, frame.excel_row)
    ]
    return frame


def read_records(path, sheet_name='Sheet1'):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError('Set DATA_DIR to the folder containing both input files: ' + str(path))
    frame = pd.read_excel(path, sheet_name=sheet_name, dtype=str, keep_default_na=False,
                          usecols=lambda c: c in METADATA_COLUMNS + TRIAGE_COLUMNS)
    if 'Article Title' not in frame or 'Abstract' not in frame:
        raise ValueError('Expected Article Title and Abstract columns in ' + str(path))
    return prepare_records(frame, path)


def match_after_to_before(before, after):
    """Match DOI first, then WOS ID, then an unambiguous normalized title.

    Duplicate rows sharing a DOI resolve to the same paper. Conflicting IDs
    remain unmatched; a supplied, different DOI is never silently replaced.
    """
    maps = {}
    for column in ['doi_key', 'wos_key', 'title_key']:
        maps[column] = before.loc[before[column].ne('')].groupby(column)['paper_id'].agg(set).to_dict()
    known_dois = before.groupby('paper_id')['doi_key'].agg(lambda s: set(s) - {''}).to_dict()
    before_ids = set(before.paper_id)
    records = []
    for _, row in after.iterrows():
        candidates = [(col, maps[col].get(row[col], set()))
                      for col in ['doi_key', 'wos_key', 'title_key'] if row[col]]
        hits = [(col, ids) for col, ids in candidates if ids]
        chosen, method, issue = None, 'unmatched', ''
        for col, ids in hits:
            if len(ids) == 1:
                chosen, method = next(iter(ids)), col.replace('_key', '')
                break
        if chosen is not None:
            # IDs are stronger than titles, but disagreement in either is visible.
            conflicts = [col for col, ids in hits if chosen not in ids]
            if row.doi_key and known_dois[chosen] and row.doi_key not in known_dois[chosen]:
                conflicts.append('different DOI')
            if conflicts:
                chosen, method, issue = None, 'conflict', '; '.join(conflicts)
        elif hits:
            method, issue = 'ambiguous', 'More than one before-paper matches the available identifiers.'
        paper_id = chosen if chosen is not None else row.paper_id
        if chosen is None and paper_id in before_ids:
            paper_id = 'unmatched:' + paper_id
        records.append((paper_id, chosen is not None, method, issue))
    out = after.copy()
    out[['paper_id', 'matched_to_before', 'match_method', 'match_issue']] = pd.DataFrame(
        records, index=after.index, columns=['paper_id', 'matched_to_before', 'match_method', 'match_issue'])
    return out


def unique_papers(records):
    """One representative per DOI (fallback WOS/title); retain source row provenance."""
    out = records.copy()
    out['_text_length'] = out['Article Title'].str.len() + out['Abstract'].str.len()
    out = out.sort_values('_text_length', ascending=False, kind='stable').drop_duplicates('paper_id')
    provenance = records.groupby('paper_id', sort=False).agg(
        source_rows=('excel_row', lambda s: ', '.join(str(v) for v in s)),
        source_record_count=('excel_row', 'size'))
    return out.drop(columns='_text_length').merge(provenance, on='paper_id', validate='one_to_one')


def label_inventory(records, label_columns=TRIAGE_COLUMNS):
    rows = []
    for col in label_columns:
        if col in records:
            labels = records[col].fillna('').astype(str).str.strip().str.upper()
            for label, n in labels.value_counts(dropna=False).items():
                rows.append({'label_column': col, 'label': label or '(blank)',
                             'rows': int(n),
                             'unique_papers': records.loc[labels.eq(label), 'paper_id'].nunique()})
    return pd.DataFrame(rows)


def apply_overrides(classified, override_path, categories):
    result = classified.copy()
    result['automatic_category'] = result['category']
    result['manual_reviewed'] = False
    result['review_note'] = ''
    result['classification_method'] = 'topic rules'
    path = Path(override_path)
    if not path.exists():
        return result
    edits = pd.read_csv(path, dtype=str, keep_default_na=False)
    required = {'paper_id', 'manual_category'}
    if not required.issubset(edits):
        raise ValueError('Override CSV must have paper_id and manual_category columns.')
    edits['manual_category'] = edits.manual_category.str.strip()
    edits = edits.loc[edits.manual_category.ne('')].copy()
    if edits.paper_id.duplicated().any():
        raise ValueError('Only one manual override per paper_id is allowed.')
    invalid = set(edits.manual_category) - set(categories)
    unknown = set(edits.paper_id) - set(result.paper_id)
    if invalid or unknown:
        raise ValueError('Invalid override categories or unknown paper IDs: ' + repr((invalid, unknown)))
    if 'is_review_article' in result.columns:
        review_ids = set(result.loc[result.is_review_article.eq(True), 'paper_id'])
        incompatible = edits.loc[
            edits.paper_id.isin(review_ids)
            & edits.manual_category.ne('Theory & modeling'), 'paper_id'].tolist()
        if incompatible:
            raise ValueError(
                'Review articles must remain Theory & modeling under the requested taxonomy. '
                'Incompatible manual category override for paper IDs: ' + repr(incompatible)
                + '. If a paper is not a review, correct its genre metadata/detection before '
                'applying the category override.')
    for _, edit in edits.iterrows():
        mask = result.paper_id.eq(edit.paper_id)
        result.loc[mask, 'category'] = edit.manual_category
        result.loc[mask, 'manual_reviewed'] = True
        result.loc[mask, 'review_needed'] = False
        result.loc[mask, 'review_reason'] = ''
        result.loc[mask, 'review_note'] = edit.get('review_note', '')
        result.loc[mask, 'classification_method'] = 'manual override'
    return result


def attach_classification(records, classified):
    metadata = set(METADATA_COLUMNS + ['doi_key', 'wos_key', 'title_key', 'source_file',
                                       'excel_row', 'source_rows', 'source_record_count',
                                       'match_method', 'match_issue', 'matched_to_before',
                                       'selected_Y', 'in_before', 'after_selected_Y'])
    annotation_cols = [c for c in classified if c not in metadata and c not in TRIAGE_COLUMNS]
    return records.merge(classified[annotation_cols], on='paper_id', how='left', validate='many_to_one')


def count_categories(before_classified, after_y_classified, categories, unique=True, subset=True,
                     full_decisions=None):
    """Count topics, optionally splitting the full corpus by explicit decisions.

    With full_decisions, Y/N/Unknown/Conflict partition the same before population.
    The selected workbook validates the Y paper identities and their categories;
    row-mode split counts use before-source multiplicities, not selected-file
    multiplicities. Missing or conflicting decisions are never inferred to be N.
    Calls without full_decisions retain the original before/selected-Y behavior.
    """
    if before_classified.groupby('paper_id').category.nunique(dropna=False).gt(1).any():
        raise ValueError('Duplicate before rows must share one category per paper.')
    if after_y_classified.groupby('paper_id').category.nunique(dropna=False).gt(1).any():
        raise ValueError('Duplicate selected rows must share one category per paper.')
    before = before_classified.drop_duplicates('paper_id') if unique else before_classified
    after = after_y_classified.drop_duplicates('paper_id') if unique else after_y_classified
    canonical_categories = before_classified.drop_duplicates('paper_id').set_index('paper_id').category
    matched_after = after.loc[after.paper_id.isin(canonical_categories.index)]
    if not matched_after.category.eq(matched_after.paper_id.map(canonical_categories)).all():
        raise ValueError('Selected papers must reuse their before-triage category.')
    split = None
    if full_decisions is not None:
        if not full_decisions.paper_id.is_unique:
            raise ValueError('Full decisions require one row per paper.')
        if set(full_decisions.paper_id) != set(before.paper_id):
            raise ValueError('Full decisions must cover exactly the before-triage paper IDs.')
        if not full_decisions.triage_decision.isin(['Y', 'N', 'Unknown', 'Conflict']).all():
            raise ValueError('Full decisions must use Y, N, Unknown, or Conflict.')
        if 'category' in full_decisions and not full_decisions.category.eq(
                full_decisions.paper_id.map(canonical_categories)).all():
            raise ValueError('Full decisions must reuse their before-triage category.')
        explicit_y = set(full_decisions.loc[full_decisions.triage_decision.eq('Y'), 'paper_id'])
        if explicit_y != set(after.paper_id):
            raise ValueError('Selected Y paper IDs differ from explicit full-corpus Y decisions.')
        decisions = before.paper_id.map(full_decisions.set_index('paper_id').triage_decision)
        split = {label: before.loc[decisions.eq(label)]
                 for label in ['Y', 'N', 'Unknown', 'Conflict']}
        after = split['Y']
    order = list(categories) + ['Unclassified']
    result = pd.DataFrame(index=pd.Index(order, name='Category'))
    result['Before triage'] = before.category.value_counts().reindex(order, fill_value=0)
    result['After triage (Y)'] = after.category.value_counts().reindex(order, fill_value=0)
    if split is not None:
        for label in ['N', 'Unknown', 'Conflict']:
            result['After triage (' + label + ')'] = split[label].category.value_counts().reindex(order, fill_value=0)
    result['Before needing review'] = before.groupby('category').review_needed.sum().reindex(order, fill_value=0)
    result['After Y needing review'] = after.groupby('category').review_needed.sum().reindex(order, fill_value=0)
    if split is not None:
        result['After N needing review'] = split['N'].groupby('category').review_needed.sum().reindex(order, fill_value=0)
    result = result.astype(int)
    result.loc['TOTAL'] = result.sum()
    result['Before share (%)'] = (100 * result['Before triage'] / len(before)).round(2) if len(before) else float('nan')
    result['After Y share (%)'] = (100 * result['After triage (Y)'] / len(after)).round(2) if len(after) else float('nan')
    if split is not None:
        result['After N share (%)'] = (100 * result['After triage (N)'] / len(split['N'])).round(2) if len(split['N']) else float('nan')
        assert result[['After triage (' + label + ')' for label in split]].sum(axis=1).eq(result['Before triage']).all()
    if subset and unique:
        result['Retained (%)'] = (100 * result['After triage (Y)'] /
                                  result['Before triage'].replace(0, float('nan'))).round(2)
    assert result.loc['TOTAL', 'Before triage'] == len(before)
    assert result.loc['TOTAL', 'After triage (Y)'] == len(after)
    if unique and subset:
        assert set(after.paper_id).issubset(set(before.paper_id))
        assert result['After triage (Y)'].le(result['Before triage']).all()
    return result


def attach_full_decisions(before_papers, decisions, column='Agent_YN'):
    """Attach explicit decisions without inventing N for missing/ambiguous records.

    before_papers must contain one row per canonical paper ID. The decisions
    input has been identity-matched against the full corpus. Mixed Y/N duplicate
    records remain Conflict, and blanks/unsupported labels remain Unknown.
    """
    if not before_papers.paper_id.is_unique:
        raise ValueError('Expected one row per before paper.')
    if column not in decisions:
        raise ValueError('Missing explicit decision column: ' + column)
    mapped = decisions.loc[decisions.matched_to_before.astype(bool)].copy()
    mapped['_decision'] = mapped[column].fillna('').astype(str).str.strip().str.upper()

    def consensus(values):
        labels = set(values) - {''}
        if labels == {'Y'}:
            return 'Y'
        if labels == {'N'}:
            return 'N'
        if 'Y' in labels and 'N' in labels:
            return 'Conflict'
        return 'Unknown'

    decision = mapped.groupby('paper_id')['_decision'].agg(consensus)
    result = before_papers.copy()
    result['triage_decision'] = result.paper_id.map(decision).fillna('Unknown')
    result['decision_column'] = column
    result['decision_source'] = result.paper_id.map(
        mapped.groupby('paper_id').source_file.first()).fillna('')
    return result


def count_yn(papers, categories):
    """Disjoint unique-paper Y/N groups, with unresolved decisions explicit."""
    if not papers.paper_id.is_unique:
        raise ValueError('Y/N counts require unique papers.')
    order = list(categories) + ['Unclassified']
    out = pd.crosstab(papers.category, papers.triage_decision).reindex(
        index=order, columns=['Y', 'N', 'Unknown', 'Conflict'], fill_value=0).astype(int)
    out.index.name = 'Category'
    out.columns.name = None
    out['Before total'] = out.sum(axis=1)
    out.loc['TOTAL'] = out.sum()
    out['Y among decided (%)'] = (100 * out.Y / (out.Y + out.N).replace(0, float('nan'))).round(2)
    assert out.loc['TOTAL', 'Before total'] == len(papers)
    return out

In [ ]:
before_rows = read_records(BEFORE_PATH, SHEET_NAME)
after_rows = read_records(AFTER_PATH, SHEET_NAME)
decision_rows = read_records(DECISION_PATH, SHEET_NAME)
decision_rows = match_after_to_before(before_rows, decision_rows)
if LABEL_COLUMN not in after_rows:
    raise ValueError('Missing LABEL_COLUMN. Available triage columns: ' +
                     repr([c for c in TRIAGE_COLUMNS if c in after_rows]))
after_rows = match_after_to_before(before_rows, after_rows)
after_rows['selected_Y'] = is_y(after_rows[LABEL_COLUMN])
after_y_rows = after_rows.loc[after_rows.selected_Y].copy()

coverage = pd.DataFrame([
    {'Dataset': name, 'Spreadsheet rows': len(frame), 'Unique papers': frame.paper_id.nunique(),
     'Extra duplicate rows': len(frame) - frame.paper_id.nunique(),
     'Missing title': frame['Article Title'].map(normalize_text).eq('').sum(),
     'Missing abstract': frame['Abstract'].map(normalize_text).eq('').sum(),
     'Missing DOI': frame.doi_key.eq('').sum()}
    for name, frame in [('Before: Full', before_rows), ('Selected: all rows', after_rows),
                        ('After: ' + LABEL_COLUMN + '=Y', after_y_rows)]
])
display(coverage)
print('Actual label counts (different label groups may share duplicate paper IDs):')
display(label_inventory(after_rows))
display(after_rows.groupby(['match_method', 'matched_to_before'], dropna=False).size().rename('Rows').to_frame())

labels_normalized = after_rows[LABEL_COLUMN].str.strip().str.upper()
label_conflicts = after_rows.assign(normalized_decision=labels_normalized).groupby('paper_id').normalized_decision.agg(
    lambda s: '; '.join(sorted(set(s) - {''})))
label_conflicts = label_conflicts.loc[label_conflicts.str.contains(';', regex=False)].reset_index()
print('Papers with conflicting nonblank decisions:', len(label_conflicts))
if len(label_conflicts):
    display(label_conflicts.head(20))
unmatched_y = after_y_rows.loc[~after_y_rows.matched_to_before.astype(bool)]
print('Y rows not safely matched to Full:', len(unmatched_y))
if len(unmatched_y):
    display(unmatched_y[['excel_row', 'DOI', 'Article Title', 'match_method', 'match_issue']].head(20))
print('Explicit full-corpus decisions:')
display(label_inventory(decision_rows, [DECISION_COLUMN]))
print('Unmatched decision records:', int((~decision_rows.matched_to_before.astype(bool)).sum()))

## 3. Topic rubric and transparent classifier

| Display category | Assignment rule |
|---|---|
| Chemical synthesis | Synthetic routes, molecular/ligand synthesis, or post-synthetic transformations as before; also original papers explicitly reporting experimental MOF/coordination-framework preparation, including synthesis-and-characterization and synthesis-and-application reports. |
| Theory & modeling | All identified review articles (priority rule), plus original work with computational/theoretical methods, simulation, modeling, or prediction as the central topic. |
| Crystal engineering | Structural characterization, topology, packing, polymorphism, and assembly without an explicit framework-preparation statement qualifying for the synthesis-inclusive group. |
| Functional materials | Applications or properties without an explicit framework-preparation statement qualifying for the synthesis-inclusive group, including use of a MOF catalyst to make molecular products. |

For the preserved primary topic, titles receive more weight than abstracts. Distinct phrase groups contribute capped
scores, so repeating a word does not increase its score indefinitely. Specific application
evidence should outweigh a routine mention of synthesis; auxiliary DFT should not
automatically make an experimental application paper theoretical. MOF/material/crystal
mentions alone are weak evidence. Broader lexical fallback assignments are explicitly
flagged as low confidence and retain their matching terms; any unsupported domain
default is separately marked. No triage label is used to choose a topic.

The original primary-topic category and scores remain available. The broader display
assignment has a separate reason and the exact framework-preparation evidence. Reviews
retain their priority rule. Computational, background, hypothetical, and negated
preparation claims do not establish experimental framework synthesis. Missing abstract
evidence does not prove synthesis was absent from the full paper.

The rules below expose their phrases, scores, selection logic, and review criteria.
**Scores and margins are heuristic evidence strengths, not calibrated probabilities.**
The four topics overlap, and this automatic procedure does not fully understand a paper's
main contribution. The review queue and editable overrides are part of the workflow.

In [ ]:
"""Topic classifier v5: synthesis-inclusive grouping with a v4 topic audit.

The public category is synthesis-inclusive: an original paper with an explicit,
affirmative preparation claim for a MOF/coordination framework, network, or
polymer is Chemical synthesis, including routine preparation in structure and
application papers. Review articles retain Theory & modeling priority. Generic
material preparation, use of an existing MOF catalyst, derivative-only synthesis,
and background, hypothetical, or computational preparation are not such claims.
This is an additive policy; v4 molecular/method/post-synthetic Chemical synthesis
assignments are retained. The v4 primary topic, reason, evidence, scores and
diagnostics are preserved independently of the public grouping. The evidence
rule is specified without reading or fitting any triage Y/N decisions.

The retained v4 primary-topic policy follows:

The four primary topics overlap. Chemical synthesis means developing synthetic
routes, molecular/ligand synthesis, or post-synthetic chemistry. Using a MOF or
other solid as a catalyst to make a molecular product is Functional materials;
the presence of a reaction name alone is not synthesis of the framework.
Crystal engineering means
constructing/understanding coordination structures, networks, and their topology;
new-framework synthesis plus substantial structural analysis belongs here.
Functional materials means application/property performance, including derived
materials. Theory & modeling means computation or theory is the main study.

Phrase groups provide a fallback; contribution rules add evidence from title
grammar and individual abstract sentences. An application studied primarily by
simulation remains Theory & modeling. Supporting calculations alone do not
override an experimental application. Hydrothermal/solvothermal conditions and
generic synthesis are weak evidence. Construction/structure reports with a
brief final luminescence/magnetism measurement can be Crystal engineering.

Title group scores are capped at 20/category and abstract scores at 6/category.
Explicit primary-contribution evidence adds 30 points to the selected topic.
Scores are evidence bookkeeping, NOT probabilities or measured accuracy.
Conflicting/mixed contributions, weak evidence, and missing data are flagged.
If specific rules find no evidence, broad lexical evidence assigns a provisional
topic and records its matched words, confidence, and fallback status. With text
but no lexical evidence, the corpus-domain default is Crystal engineering,
explicitly marked unsupported. Empty title AND abstract remain Unclassified.
The rules are heuristic and need human validation before scientific reporting.
Review articles have priority under the user's taxonomy, regardless of their
covered topic. Document Type metadata and explicit title/abstract genre cues
identify reviews; generic 'recent advances' or 'perspective' alone are not enough.
Only title, abstract, and optional document type are accepted; triage Y/N labels are NEVER inputs and
the method is not fitted to any desired category proportions. Python >= 3.8.
"""

import math
import re
import unicodedata


CATEGORIES = (
    "Chemical synthesis",
    "Theory & modeling",
    "Crystal engineering",
    "Functional materials",
)
UNCLASSIFIED = "Unclassified"
CLASSIFIER_VERSION = "5.0.0"
TAXONOMY_POLICY = "synthesis-inclusive-v1"

_SCORE_KEYS = (
    "score_chemical_synthesis",
    "score_theory_modeling",
    "score_crystal_engineering",
    "score_functional_materials",
)

# Each group is (name, title base weight, abstract weight, regex alternatives).
# Alternatives in the same group cannot accumulate points by repetition.
_GROUP_DEFINITIONS = {
    CATEGORIES[0]: [
        ("synthetic routes", 3.0, 1.7, [
            r"mechanochem\w*",
            r"electrochemical synthes\w*", r"electrosynth\w*",
            r"microwave (?:assisted )?synth\w*", r"sonochemical\w*",
            r"solvent free synthes\w*", r"solventless synthes\w*",
            r"solid state synthes\w*", r"vapor (?:phase )?synth\w*",
            r"vapour (?:phase )?synth\w*", r"flow synthes\w*",
            r"continuous flow", r"room temperature synthes\w*",
            r"ambient (?:condition\w* )?synth\w*", r"ball milling",
            r"microfluidic\w*", r"layer by layer (?:growth|deposition|synthes\w*)",
            r"(?:controlled|controllable|modular|epitaxial|colloid assisted) growth",
            r"(?:emulsion|surfactant|soft|hard) templat\w*",
        ]),
        ("routine synthesis conditions (weak)", 0.45, 0.1, [
            r"solvothermal\w*", r"hydrothermal\w*",
        ]),
        ("synthesis development", 2.6, 1.8, [
            r"synthetic (?:route\w*|method\w*|strateg\w*|protocol\w*|approach\w*)",
            r"(?:scalable|scale up|large scale|high yield|rapid|one pot|one step|green|controlled|controllable) synthes\w*",
            r"(?:synthesis|synthetic) (?:optimization|optimisation|kinetics|mechanism\w*)",
            r"(?:modulated|modulator assisted|template assisted|seed mediated) synthes\w*",
            r"(?:nucleation|formation) mechanism\w*",
            r"(?:design|synthesis) of (?:complex )?organic (?:building blocks?|ligands?)",
        ]),
        ("post-synthetic chemistry", 3.2, 1.9, [
            r"post\s?synthetic(?:ally)? (?:\w+ ){0,2}(?:modification\w*|modified|functionalization\w*|functionalisation\w*|exchange\w*|metalation\w*|metallation\w*)",
            r"(?:linker|ligand) exchange", r"transmetalation\w*",
            r"transmetallation\w*", r"covalent functionalization\w*",
            r"covalent functionalisation\w*",
            r"(?:ligand|linker) (?:synthes\w*|functionalization\w*|functionalisation\w*)",
            r"(?:grafting|grafted) (?:from|onto|on)\b",
        ]),
        ("molecular reaction development", 3.3, 1.8, [
            r"(?:c h|c c|c n|c o) (?:bond )?(?:activation\w*|functionalization\w*|functionalisation\w*|formation|coupling)",
            r"(?:suzuki|heck|sonogashira|buchwald hartwig|knoevenagel|hantzsch|biginelli|aldol|pechmann)\w*(?: \w+){0,2} (?:coupling|reaction|condensation)",
            r"(?:cross coupling|cycloaddition|esterification|transesterification|amidation|acetalization|acetalisation|alkylation|arylation)",
            r"(?:total|asymmetric|enantioselective|stereoselective) synthes\w*",
            r"(?:oxidation|hydrogenation) of (?:\w+ ){0,3}(?:alcohol\w*|aldehyde\w*|alkene\w*|alkyne\w*|amine\w*|nitroarene\w*)",
            r"synthes\w* of (?:\w+ ){0,3}cyclic carbonate\w*",
            r"(?:olefination|annulation|cyanosilylation|dimerization|dimerisation)",
            r"(?:cross dehydrogenative|cross dehydrogenation) coupling",
            r"(?:intramolecular|intermolecular) (?:\w+ ){0,4}rearrangement\w*",
            r"nucleophilic aromatic substitution",
        ]),
        ("generic synthesis report (weak)", 0.35, 0.1, [
            r"synthes(?:is|es)", r"preparation and characterization",
            r"preparation and characterisation",
        ]),
    ],
    CATEGORIES[1]: [
        ("explicit computational focus", 3.8, 2.6, [
            r"(?:computational|theoretical|atomistic|in silico) (?:study|studies|investigation\w*|analysis|screening|prediction\w*|design|approach\w*|insight\w*|model\w*|assessment)",
            r"(?:computer|computational|molecular|atomistic|multiscale|quantum) simulation\w*",
            r"(?:computational|theoretical) chem\w*",
        ]),
        ("quantum electronic calculations", 3.0, 0.55, [
            r"density functional(?: theory)?", r"dft", r"ab initio",
            r"first principles?", r"quantum chemical (?:calculation\w*|model\w*)",
            r"qm mm", r"quantum mechanics and molecular mechanics",
        ]),
        ("molecular and statistical simulations", 3.0, 0.7, [
            r"molecular dynamics", r"monte carlo", r"gcmc",
            r"force field\w*", r"statistical mechanic\w*",
            r"coarse grained (?:simulation\w*|model\w*)",
            r"molecular (?:modeling|modelling)",
        ]),
        ("predictive and mathematical methods", 3.2, 1.8, [
            r"machine learning", r"deep learning", r"neural network\w*",
            r"graph neural", r"data driven", r"mathematical model\w*",
            r"(?:kinetic|thermodynamic|continuum|multiscale) (?:modeling|modelling)",
            r"high throughput computational", r"virtual screening",
        ]),
    ],
    CATEGORIES[2]: [
        ("network topology and assembly", 3.3, 2.0, [
            r"topolog\w*", r"reticular (?:chemistry|design|synthesis)",
            r"isoreticular\w*", r"interpenetrat\w*", r"catenat\w*",
            r"supramolecular (?:assembl\w*|architectur\w*|synthons?|networks?|chemistry)",
            r"(?:directed|hierarchical|coordination driven) self assembl\w*",
            r"(?:network|framework) (?:connectivity|catenation)",
        ]),
        ("crystal packing and structural design", 3.2, 1.7, [
            r"crystal engineering", r"crystal packing", r"crystal structure\w*",
            r"(?:structure|structural) directing", r"supramolecular synthon\w*",
            r"polymorph\w*", r"co\s?crystal\w*", r"packing motif\w*",
            r"structural diversit\w*", r"coordination geometr\w*",
            r"coordination modes?", r"(?:homo)?chiral (?:\w+ ){0,3}(?:framework\w*|coordination polymer\w*)",
            r"(?:secondary|second) building units?", r"(?:framework|pore) engineering",
            r"(?:hydrogen|halogen) bond\w* (?:network\w*|assembl\w*)",
        ]),
        ("crystallization and crystal growth", 3.0, 1.8, [
            r"crystalli[sz]ation", r"crystal growth", r"single crystal to single crystal",
            r"single crystal transformation\w*", r"crystal nucleation",
            r"nucleation and growth", r"crystal habit\w*",
            r"(?:crystalline to amorphous|amorphous amorphous|crystal phase) transition\w*",
        ]),
    ],
    CATEGORIES[3]: [
        ("adsorption, storage, and separation", 3.0, 1.8, [
            r"adsor\w*", r"sorption", r"separat\w*", r"membrane\w*",
            r"(?:gas|hydrogen|methane|carbon dioxide|co2) (?:storage|capture|uptake)",
            r"desalination", r"water harvesting", r"ion exchange",
        ]),
        ("energy conversion and electrochemistry", 3.0, 1.9, [
            r"batter\w*", r"supercapacitor\w*", r"capacitor\w*",
            r"electrode\w*", r"electrolyte\w*", r"fuel cell\w*",
            r"energy (?:storage|conversion)", r"solar cell\w*", r"photovoltaic\w*",
            r"(?:hydrogen|oxygen) (?:evolution|production|generation|reduction)",
            r"water splitting", r"electrocatal\w*",
            r"(?:co2|carbon dioxide) (?:reduction|photoreduction|hydrogenation)",
        ]),
        ("catalytic and environmental performance", 2.5, 1.5, [
            r"photocatal\w*", r"heterogeneous catal\w*",
            r"catalytic (?:performance|activity|application\w*)",
            r"(?:enzyme|protein) immobilization", r"(?:enzyme|protein) immobilisation",
            r"(?:nerve agent\w*|chemical warfare|detoxification)",
            r"(?:pollutant\w*|dye\w*) (?:removal|degradation|adsorption)",
            r"(?:removal|degradation) of (?:\w+ ){0,3}(?:pollutant\w*|dye\w*|antibiotic\w*)",
            r"water (?:purification|remediation|treatment)", r"wastewater",
        ]),
        ("sensing and optical function", 3.0, 1.6, [
            r"sens(?:ing|or\w*)", r"detect(?:ion|ing) of", r"chemosens\w*",
            r"fluorescen\w*", r"luminescen\w*", r"photoluminescen\w*",
            r"phosphorescen\w*", r"light emitting", r"photophysic\w*",
            r"nonlinear optic\w*", r"optical propert\w*",
        ]),
        ("electronic, magnetic, and mechanical function", 3.0, 1.6, [
            r"(?:proton|ion|ionic|electrical|electronic|thermal) conduct\w*",
            r"(?:charge|electron|proton|ion) transport", r"semiconduct\w*",
            r"magnet\w*", r"spin crossover", r"ferroelectric\w*",
            r"piezoelectric\w*", r"dielectric\w*", r"thermoelectric\w*",
            r"mechanical propert\w*", r"elastic propert\w*",
            r"electronic propert\w*", r"electronic device\w*",
            r"thermal expansion", r"thermomechanic\w*",
        ]),
        ("biomedical and delivery applications", 3.0, 1.8, [
            r"drug (?:delivery|release|loading)", r"antimicrobial\w*",
            r"antibacterial\w*", r"anticancer\w*", r"bioimag\w*",
            r"photodynamic therap\w*", r"photothermal therap\w*",
            r"cancer (?:therap\w*|treatment)", r"wound heal\w*",
            r"cellular uptake", r"cytotoxic\w*", r"theranostic\w*",
            r"biomedic\w*", r"biosens\w*", r"biocompatib\w*",
            r"antitumou?r\w*", r"tumou?r (?:therap\w*|treatment|targeting)",
            r"bioactiv\w*", r"biocid\w*", r"biocatal\w*",
        ]),
    ],
}


def _compile(patterns):
    return re.compile(r"\b(?:" + "|".join(patterns) + r")\b", re.I)


_GROUPS = {
    category: [(name, title_weight, abstract_weight, _compile(patterns))
               for name, title_weight, abstract_weight, patterns in groups]
    for category, groups in _GROUP_DEFINITIONS.items()
}
_COMPUTATIONAL_TITLE = _compile([
    r"(?:computational|theoretical|atomistic|in silico) (?:study|studies|investigation\w*|analysis|screening|prediction\w*|design|approach\w*|insight\w*|model\w*|assessment)",
    r"(?:computer|computational|molecular|atomistic|multiscale|quantum) simulation\w*",
    r"density functional(?: theory)?", r"dft", r"ab initio",
    r"first principles?", r"molecular dynamics", r"monte carlo", r"gcmc",
    r"machine learning", r"deep learning", r"neural network\w*",
    r"molecular (?:modeling|modelling)", r"data driven",
    r"qm mm", r"quantum mechanics and molecular mechanics",
])
_PURPOSE_MARKER = re.compile(r"\b(?:for|towards?|as)\b")
_EXPERIMENTAL_TITLE = re.compile(r"\bexperimental\w*\b")
_COMPUTATIONAL_METHOD = _compile([
    r"computational\w*", r"theoretical\w*", r"simulat\w*",
    r"density functional(?: theory)?", r"dft", r"ab initio",
    r"first principles?", r"monte carlo", r"gcmc", r"molecular dynamics",
    r"machine learning", r"neural network\w*", r"quantum chemical",
    r"qm mm", r"quantum mechanics and molecular mechanics",
])
_STUDY_ACTION = _compile([
    r"investigat\w*", r"stud(?:y|ies|ied)", r"screen\w*", r"predict\w*",
    r"simulat\w*", r"calculat\w*", r"understanding", r"explor\w*",
    r"examin\w*", r"evaluat\w*", r"assess\w*", r"develop\w*",
])
_SELF_REPORT = _compile([
    r"we", r"herein", r"here", r"(?:this|present) (?:work|study|paper)",
    r"in this (?:article|research)",
])
_PRIMARY_COMPUTATION = _compile([
    r"(?:we|herein we|here we) (?:have )?(?:computationally|theoretically) \w+",
    r"(?:we|this work|this study|present work|present study) (?:\w+ ){0,7}(?:computational|theoretical) (?:study|investigation|understanding|screening|analysis|approach)",
    r"(?:we|this work|this study|present work|present study) (?:\w+ ){0,6}(?:perform\w*|conduct\w*|use\w*|employ\w*) (?:\w+ ){0,7}(?:simulat\w*|density functional|dft|monte carlo|machine learning)",
    r"(?:studied|investigated|examined|evaluated|predicted|screened) (?:\w+ ){0,8}(?:using|by|via|with) (?:\w+ ){0,5}(?:simulat\w*|density functional|dft|monte carlo|ab initio|first principles)",
    r"(?:computational|theoretical) (?:study|investigation|screening|analysis|understanding)",
])
_SUPPORTING_COMPUTATION = _compile([
    r"corroborat\w*", r"rationali[sz]\w*", r"support(?:ing|ed|s)?",
    r"confirm\w*", r"also", r"in addition", r"furthermore", r"moreover",
    r"(?:explain|interpret|elucidate|understand) (?:\w+ ){0,5}(?:observed|experimental|measured)",
    r"(?:consistent|agreement) with (?:\w+ ){0,3}experiment\w*",
])
_SYNTHESIS_VERB = _compile([
    r"synthesi[sz](?:ed|ing)", r"prepared", r"fabricated", r"assembled",
    r"obtained", r"grown", r"constructed",
])
_REPORTED_SYNTHESIS = _compile([
    r"(?:we|herein|here) (?:\w+ ){0,8}(?:synthesi[sz]\w*|prepar\w*|fabricat\w*|assembl\w*|construct\w*)",
    r"(?:has|have|was|were|is|are) (?:\w+ ){0,4}(?:synthesized|synthesised|prepared|fabricated|assembled|constructed)",
    r"(?:new|novel) (?:\w+ ){0,9}(?:synthesized|synthesised|prepared|obtained|constructed)",
])
_FRAMEWORK_SUBJECT = _compile([
    r"coordination (?:polymer\w*|network\w*|framework\w*|compound\w*|complex\w*)",
    r"metal organic (?:framework\w*|network\w*)", r"mofs?",
    r"supramolecular", r"zeolitic imidazolate framework\w*",
])
_STRUCTURAL_TITLE = _compile([
    r"crystal structures?", r"structures?", r"structural (?:diversity|modulation|characterization|characterisation|transformation\w*)",
    r"topolog\w*", r"assembl\w*", r"packing", r"polymorph\w*",
    r"crystal engineering", r"interpenetrat\w*", r"reticular",
])
_STRUCTURAL_DETAILS = _compile([
    r"topolog\w*", r"interpenetrat\w*", r"catenat\w*",
    r"coordination (?:geometr\w*|mode\w*|environment\w*)",
    r"crystal structures?", r"structural (?:diversity|motif\w*|feature\w*)",
    r"(?:1d|2d|3d|one dimensional|two dimensional|three dimensional) (?:\w+ ){0,4}(?:net\w*|chain\w*|layer\w*|framework\w*|structure\w*)",
    r"(?:nodal|connected) (?:\w+ ){0,2}net\w*", r"crystal packing",
    r"(?:metal metal|ag ag|cu cu|au au) (?:interaction\w*|contact\w*|bond\w*)",
])
_DIFFRACTION = _compile([
    r"single crystal (?:x ray|diffraction)", r"x ray (?:single crystal|diffraction|crystallograph\w*)",
])
_NEW_STRUCTURES = _compile([
    r"(?:new|novel) (?:\w+ ){0,10}(?:coordination polymer\w*|framework\w*|coordination network\w*|complex\w*)",
    r"(?:one|two|three|four|five|six|seven|eight|nine|ten|\d+) (?:\w+ ){0,4}coordination polymer\w*",
])
# Performance/application evidence excludes generic luminescence or magnetism:
# those measurements frequently supplement a structure-centered report.
_APPLICATION = _compile([
    r"adsor\w*", r"(?:gas|hydrogen|methane|carbon dioxide|co2) (?:storage|capture|uptake|separation)",
    r"membrane\w*", r"desalination", r"water harvesting",
    r"sens(?:ing|or\w*)", r"chemosens\w*", r"detect(?:ion|ing) of",
    r"electrocatal\w*", r"photocatal\w*", r"water splitting",
    r"(?:hydrogen|oxygen) (?:evolution|production|generation|reduction)",
    r"(?:co2|carbon dioxide) (?:reduction|photoreduction|hydrogenation)",
    r"batter\w*", r"supercapacitor\w*", r"electrode\w*", r"fuel cell\w*",
    r"energy (?:storage|conversion)", r"solar cell\w*", r"photovoltaic\w*",
    r"(?:proton|ion|ionic|electrical|electronic|thermal) conduct\w*",
    r"(?:pollutant\w*|dye\w*) (?:removal|degradation)", r"wastewater",
    r"drug (?:delivery|release|loading)", r"cellular uptake", r"cytotoxic\w*",
    r"antibacterial\w*", r"antimicrobial\w*", r"bioimag\w*", r"theranostic\w*",
    r"(?:photodynamic|photothermal|cancer|tumou?r) therap\w*",
    r"thermoelectric\w*", r"biosens\w*", r"chemiresistor\w*",
    r"(?:enzyme|protein) immobili[sz]ation", r"biocatal\w*",
    r"bioactiv\w*", r"biocid\w*", r"detoxification",
    r"(?:nerve agent\w*|chemical warfare)",
    r"(?:mechanical|elastic|electronic) propert\w*",
    r"(?:catalytic|catalysis) (?:performance|activity|application\w*|abilit\w*|behavior|behaviour)",
])
_CATALYSIS = _compile([r"catalys\w*", r"catalyz\w*", r"catalyt\w*", r"catal\w*catalyst\w*"])
_SOLID_CATALYST = _compile([
    r"(?:metal organic|porous|coordination|covalent organic|metalloporphyrinic) (?:\w+ ){0,2}(?:framework\w*|network\w*|polymer\w*)",
    r"mofs?", r"cofs?", r"zeolitic imidazolate framework\w*",
    r"(?:heterogeneous|solid|porous|supported|immobilized|immobilised|polymeric) (?:\w+ ){0,3}catal\w*",
    r"nanocatal\w*", r"nanoparticle\w*", r"nanozyme\w*",
])
_CATALYTIC_OBJECTIVE = _compile([
    r"catalytic (?:performance|activity|application\w*|abilit\w*|behavior|behaviour|efficien\w*)",
    r"(?:as|over|using|with) (?:\w+ ){0,3}catalyst\w*",
    r"cataly[sz](?:ed|es|ing)", r"(?:for|in) (?:\w+ ){0,3}catalysis",
    r"catalysts? for",
])
_FRAMEWORK_REACTION_AGENT = _compile([
    r"(?:over|using|by) (?:a |an |the )?(?:\w+ ){0,3}(?:mofs?|metal organic frameworks?|porous frameworks?)",
])
_POSTSYNTHETIC_MODIFICATION = _compile([
    r"post\s?synthetic(?:ally)? (?:\w+ ){0,2}(?:modification\w*|modified|exchange\w*|transformation\w*|functionalization\w*|functionalisation\w*)",
])
_REPORTED_CATALYST_USE = _compile([
    r"(?:used|applied|employed|utili[sz]ed|tested|evaluated) (?:\w+ ){0,3}(?:as|for) (?:\w+ ){0,3}catal\w*",
    r"catalytic (?:activity|performance|ability|efficiency) (?:\w+ ){0,5}(?:measured|studied|evaluated|demonstrated)",
    r"(?:exhibited|exhibits|displayed|displays|showed|shows) (?:\w+ ){0,3}catalytic (?:activity|performance|ability|efficiency)",
])
_PERFORMANCE = _compile([
    r"performance", r"selectivit\w*", r"capacit\w*", r"efficien\w*",
    r"detection limit", r"sensitivity", r"uptake", r"cytotoxic\w*",
    r"rate", r"cycling", r"conversion", r"yield", r"activity",
])
_METHOD_DEVELOPMENT = _compile([
    r"synthetic (?:route\w*|method\w*|strateg\w*|protocol\w*|approach\w*)",
    r"(?:synthesis|synthetic) (?:optimization|optimisation|kinetics|mechanism\w*)",
    r"(?:optimi[sz]\w*|controll\w*) (?:\w+ ){0,5}synthes\w*",
    r"(?:nucleation|formation|growth) mechanism\w*",
    r"post\s?synthetic(?:ally)? (?:\w+ ){0,2}(?:modification\w*|modified|exchange\w*|transformation\w*|functionalization\w*|functionalisation\w*)",
    r"(?:linker|ligand) exchange",
])
_REVIEW = _compile([
    r"(?:this|the present) review", r"we review", r"review article",
    r"(?:recent|latest) advances", r"perspective", r"mini review",
])

# Last-resort lexical evidence is intentionally broader than the contribution
# rules. It never supplies a +30 contribution bonus, and every use requires
# review. Distinct title matches receive two points, abstract matches one.
# No training examples, paper identifiers, or Y/N labels are consulted.
_FALLBACK_TERMS = {
    CATEGORIES[0]: _compile([
        r"synthesi[sz]\w*", r"synthes\w*", r"prepar\w*", r"fabricat\w*",
        r"functionaliz\w*", r"functionalis\w*", r"grafting", r"etching",
        r"precursor\w*", r"reaction conditions?", r"ligand formation",
        r"formation mechanism\w*", r"printing", r"carbonization", r"carbonisation",
        r"rearrange\w*", r"cycliz\w*", r"cyclis\w*", r"condensation",
    ]),
    CATEGORIES[1]: _compile([
        r"simulat\w*", r"computation\w*", r"theoretic\w*", r"calculat\w*",
        r"modeling", r"modelling", r"quantum", r"statistical mechanic\w*",
    ]),
    CATEGORIES[2]: _compile([
        r"structur\w*", r"crystall\w*", r"coordination", r"network\w*",
        r"chiral\w*", r"assembl\w*", r"topolog\w*", r"porosity",
        r"unit cell", r"space group", r"diffraction", r"building blocks?",
        r"dimensional", r"[123]d", r"entangle\w*", r"framework\w*",
    ]),
    CATEGORIES[3]: _compile([
        r"catal\w*", r"uptake", r"encapsulat\w*", r"immobili[sz]\w*",
        r"capacitance", r"conductiv\w*", r"bioactiv\w*", r"biocid\w*",
        r"sorb\w*", r"adsorb\w*", r"application\w*", r"performance",
        r"sensing", r"detox\w*", r"thermomechanic\w*", r"thermal expansion",
        r"delivery", r"release", r"dechlorination", r"redox", r"capture",
    ]),
}


def _fallback_topic(title_text, abstract_text):
    """Return a provisional assignment and reproducible lexical audit trail."""
    scores = {}
    evidence = []
    for category, pattern in _FALLBACK_TERMS.items():
        title_hits = sorted(set(m.group(0) for m in pattern.finditer(title_text)))
        abstract_hits = sorted(set(m.group(0) for m in pattern.finditer(abstract_text)))
        scores[category] = 2 * len(title_hits) + len(abstract_hits)
        for field, hits in (("title", title_hits), ("abstract", abstract_hits)):
            if hits:
                evidence.append("{} [{}; broad lexical fallback]: {}".format(category, field, ", ".join(hits)))
    # Resolve exact ties by the declared order. These are deliberately NOT
    # merged into the specific phrase-group scores, which remain auditable.
    winner = max(CATEGORIES, key=lambda category: scores[category])
    if not scores[winner]:
        return CATEGORIES[2], "corpus-domain default", "No lexical evidence; provisional Crystal engineering domain default."
    return winner, "broad lexical evidence", " | ".join(evidence)


def _normalize(value):
    """Handle spreadsheet blanks without converting float NaN to evidence."""
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return ""
    value = unicodedata.normalize("NFKC", str(value)).lower()
    if value.strip() in ("nan", "none", "<na>", "nat"):
        return ""
    value = re.sub(r"[-\u2010-\u2015\u2212/]", " ", value)
    return re.sub(r"\s+", " ", value).strip()


def _abstract_sentences(abstract):
    if not _normalize(abstract):
        return []
    return [(sentence.strip(), _normalize(sentence)) for sentence in
            re.split(r"(?<=[.!?])\s+", str(abstract)) if sentence.strip()]


def _contribution_signals(title_text, abstract):
    """Find stated contributions; never infer theory from missing experiments."""
    sentences = _abstract_sentences(abstract)
    computation = []
    supporting_computation = []
    synthesis = []
    structure = []
    application = []
    method = []
    computation_result = _compile([
        r"(?:calculations?|simulations?) (?:\w+ ){0,3}(?:show\w*|reveal\w*|indicate\w*|predict\w*)",
        r"(?:calculations?|simulations?) (?:were|are|have been) (?:performed|conducted|carried out|used)",
    ])
    background_synthesis = _compile([
        r"previous\w*", r"reported by", r"in the literature",
        r"can be (?:synthesized|prepared)", r"could be (?:synthesized|prepared)",
        r"experimentally (?:reported|synthesized|synthesised) (?:mofs?|frameworks?|structures?|materials?)",
    ])
    for index, (raw, sentence) in enumerate(sentences):
        if _COMPUTATIONAL_METHOD.search(sentence):
            if _SUPPORTING_COMPUTATION.search(sentence):
                supporting_computation.append(raw)
            elif (_PRIMARY_COMPUTATION.search(sentence)
                  or (_SELF_REPORT.search(sentence) and _STUDY_ACTION.search(sentence))
                  or (index < 3 and computation_result.search(sentence))):
                computation.append(raw)
        if _REPORTED_SYNTHESIS.search(sentence) and not background_synthesis.search(sentence):
            synthesis.append(raw)
        if _STRUCTURAL_DETAILS.search(sentence):
            structure.append(raw)
        if _APPLICATION.search(sentence):
            application.append(raw)
        if _METHOD_DEVELOPMENT.search(sentence):
            method.append(raw)

    abstract_text = _normalize(abstract)
    title_synthesis = bool(_compile([r"synthes(?:is|es)", r"construction", r"assembl\w*"]).search(title_text))
    structural_title = bool(_STRUCTURAL_TITLE.search(title_text))
    framework = bool(_FRAMEWORK_SUBJECT.search(title_text + " " + abstract_text))
    new_structure = bool(_NEW_STRUCTURES.search(abstract_text))
    diffraction = bool(_DIFFRACTION.search(abstract_text))
    # XRD alone is characterization, not a structural-design contribution.
    construction_structure = (
        (title_synthesis and structural_title)
        or (bool(synthesis) and framework and
            ((new_structure and (bool(structure) or diffraction))
             or (len(structure) >= 2))))
    title_application = bool(_APPLICATION.search(title_text))
    abstract_application = any(
        _APPLICATION.search(sentence) and _PERFORMANCE.search(sentence)
        and (_SELF_REPORT.search(sentence) or index < 3)
        for index, (_, sentence) in enumerate(sentences))
    # Abstract route development needs a stated study contribution, or several
    # sentences about the route/mechanism; background language alone is weak.
    abstract_method = any(_SELF_REPORT.search(_normalize(raw)) for raw in method) or len(method) >= 2
    return {
        "computation": computation,
        "supporting_computation": supporting_computation,
        "synthesis": synthesis,
        "structure": structure,
        "application": application,
        "method": method,
        "construction_structure": construction_structure,
        "structural_title": structural_title,
        "title_application": title_application,
        "abstract_application": abstract_application,
        "abstract_method": abstract_method,
        "review_article": bool(_REVIEW.search(title_text + " " + abstract_text)
                               or re.search(r"\breview\b", title_text)),
    }


def detect_review_article(title, abstract, document_type=""):
    """Identify review genre from metadata or explicit self-description.

    Discussion of earlier reviews, peer review, or a recent advance alone does
    not establish that the current paper is a review. Keep matched evidence.
    """
    title_text = _normalize(title)
    abstract_text = _normalize(abstract)
    doc_type = _normalize(document_type)
    if re.search(r"\b(?:reviews?|mini\s?review)\b", doc_type):
        return True, "Document Type", str(document_type).strip()
    title_match = re.search(r"\b(?:review|mini\s?review)\b", title_text)
    if title_match:
        return True, "title", str(title).strip()
    genre_patterns = [
        r"\b(?:this|the present|the current|the presented|our) (?:(?:critical|comprehensive|systematic|scoping|mini|short|tutorial|literature|focused|brief|highlight|timely|presented) ){0,3}review\b",
        r"\bwe (?:(?:critically|systematically|comprehensively|briefly|specifically|will) ){0,2}review\b",
        r"\b(?:this|the present|the current) (?:paper|article|work) (?:(?:critically|systematically|comprehensively) )?reviews\b",
        r"\b(?:this|the present|the current) (?:paper|article|work) (?:provides?|presents?|offers?) (?:a |an )?(?:(?:critical|comprehensive|systematic|scoping|tutorial|literature|brief|broad|concise|detailed|up to date) ){0,3}review\b",
        r"\bwe (?:provide|present|offer|make) (?:a |an )?(?:(?:critical|comprehensive|systematic|tutorial|literature|brief|broad|concise|detailed|up to date) ){0,3}review\b",
        r"\b(?:in this|in the present|in the) (?:mini )?review\b",
        r"\b(?:a|an) (?:critical|comprehensive|systematic|literature|tutorial|brief|concise) review\b.{0,140}\b(?:is|has been) (?:presented|provided|given)\b",
        r"\breviewed in (?:this|the present) (?:paper|article|review)\b",
        r"^the review (?:focuses|considers|discusses|summarizes|summarises|covers|examines|deals|is focused)\b",
    ]
    detector = re.compile("|".join(genre_patterns), re.I)
    for raw, sentence in _abstract_sentences(abstract):
        if detector.search(sentence):
            return True, "abstract", raw
    # An overview/survey of literature can indicate review genre, whereas a
    # survey of synthesized samples/solvents is original experimental work.
    overview = re.compile(
        r"\b(?:(?:this|the present) (?:paper|article|study|work) (?:provides|presents|offers)|"
        r"we (?:provide|present|offer)) (?:a |an )?(?:(?:comprehensive|systematic|brief|critical|detailed) )?"
        r"(?:overview|survey)\b.{0,150}\b(?:literature|recent advances|developments|progress|state of the art)\b")
    own_experiments = any(_SELF_REPORT.search(sentence) and _REPORTED_SYNTHESIS.search(sentence)
                          for _, sentence in _abstract_sentences(abstract))
    if not own_experiments:
        for raw, sentence in _abstract_sentences(abstract):
            if overview.search(sentence):
                return True, "abstract literature overview", raw
    return False, "", ""


_PREPARED_FRAMEWORK = (
    r"(?:metal organic|porous coordination|coordination|zeolitic imidazolate) "
    r"(?:frameworks?|networks?|polymers?)|mofs?|"
    r"(?:uio|zif|mil|hkust|mfm|pcn|dut|nu) \d+[a-z]?"
)
_PREPARATION_PAST = (
    r"synthesi[sz]ed|prepared|fabricated|assembled|constructed|obtained|grown|crystalli[sz]ed"
)
_PREPARATION_ACTIVE = (
    r"synthesi[sz](?:e|ed)|prepar(?:e|ed)|fabricat(?:e|ed)|assembl(?:e|ed)|"
    r"construct(?:ed)?|obtain(?:ed)?|grew|grow|crystalli[sz](?:e|ed)"
)
_PREPARATION_NOUN = r"synthes(?:is|es)|preparation|fabrication|construction|assembly|growth"
_NONASSERTED_PREPARATION = re.compile(
    r"\b(?:not|never|no|cannot|can t|couldn t|didn t|without|unsuccessful|failed|"
    r"previously|previous|earlier|already|usually|typically|commonly|generally|"
    r"traditionally|conventionally|hypothetical|hypothetically|proposed|predicted|"
    r"reported by|in the literature|has been reported|have been reported|"
    r"had been|can|could|may|might|would|should|will|aim|aims|attempt|attempts)\b"
)
_DERIVED_OR_DIFFERENT_OBJECT = re.compile(
    r"\b(?:derived|based on|carbon(?:s)?|oxides?|\w*composites?|\w*hybrids?|"
    r"using|via|over|from|cataly[sz]\w*|cataly(?:sis|tic)|"
    r"products?|substrates?|molecules?|carbonates?|amines?|alcohols?|"
    r"data|results?|measurements?|properties|isotherms?|curves?)\b"
)


def _framework_preparation_evidence(title, abstract, primary_result):
    """Find local, affirmative framework-preparation claims, without label inputs.

    Matches require a preparation noun governing the framework object, a
    first-person preparation verb with that object, or a framework subject with
    a passive preparation verb. Bridges are bounded to 18 words/180 characters
    and reject object-changing prepositions and derivative/composite objects.
    Sentences, semicolons, and explicit contrast clauses are independent: a MOF
    mention elsewhere cannot make organic-product synthesis a positive claim.
    Common framework identifiers are accepted as entities. A generic 'framework'
    alone is insufficient, as it can describe a model or a supramolecular solid.
    These conservative lexical checks are evidence flags, not a parser or a
    claim of validated accuracy; ambiguous records retain the v4 assignment.
    """
    if primary_result["is_review_article"]:
        return []
    entity = r"(?P<entity>\b(?:" + _PREPARED_FRAMEWORK + r")\b)"
    bridge = r"(?P<bridge>[^.;:!?]{0,180}?)"
    passive = re.compile(entity + bridge + r"\b(?P<action>" + _PREPARATION_PAST + r")\b")
    active = re.compile(r"\b(?P<action>" + _PREPARATION_ACTIVE + r")\b" + bridge + entity)
    nominal = re.compile(r"\b(?P<action>" + _PREPARATION_NOUN + r")\b" + bridge + entity)
    evidence = []
    denied_preparation = False
    inputs = [("title", str(title).strip(), _normalize(title))] if _normalize(title) else []
    inputs += [("abstract", raw, normalized) for raw, normalized in _abstract_sentences(abstract)]
    for source, raw, normalized in inputs:
        for clause in re.split(r";|\b(?:but|whereas|however)\b", normalized):
            for relation, pattern in (("passive", passive), ("active", active), ("nominal", nominal)):
                for match in pattern.finditer(clause):
                    link = match.group("bridge")
                    if len(re.findall(r"\b\w+\b", link)) > 18:
                        continue
                    # The preparation target must be the framework, rather than
                    # a product made using it or a derivative/composite of it.
                    if _DERIVED_OR_DIFFERENT_OBJECT.search(link):
                        continue
                    entity_suffix = clause[match.end("entity"):]
                    if re.match(r"\s*(?:\([^)]*\)\s*)?"
                                r"(?:(?!(?:was|were|is|are|has|have|and|for|which|that)\b)[a-z0-9]+\s+){0,4}"
                                r"(?:derived|based|\w*composites?|\w*hybrids?)\b", entity_suffix):
                        continue
                    prefix = clause[:match.end()]
                    if _NONASSERTED_PREPARATION.search(prefix):
                        failed_attempt = re.match(
                            r"\s*(?:was|were|is|are|have been|has been) "
                            r"(?:\w+ ){0,2}(?:unsuccessful|failed)\b", clause[match.end():])
                        if source == "abstract" and (failed_attempt or re.search(
                                r"\b(?:not|never|cannot|failed|unsuccessful)\b", prefix)):
                            denied_preparation = True
                        continue
                    suffix = clause[match.end():]
                    if re.match(r"\s*(?:(?:that|which)\s+)?(?:(?:was|were|has been|have been)\s+)?"
                                r"(?:previously|earlier|reported by|in the literature)\b", suffix):
                        continue
                    # An entirely theoretical/simulation preparation statement
                    # does not establish physical framework preparation.
                    if re.search(r"\b(?:in silico|computationally|theoretically|virtually)\b", prefix):
                        continue
                    if relation == "nominal":
                        # 'Synthesis ... using a MOF' is excluded above. Require
                        # the governing 'of' and reject a new object before it.
                        if not re.match(r"\s*(?:(?:and|,|crystal|structures?|structural|"
                                        r"characterization|characterisation|isolation|properties|"
                                        r"luminescence|magnetic|photoluminescent)\s*)*of\b", link):
                            continue
                        if re.search(r"\b(?:in|within|inside|into|on|for|with|by|containing|"
                                     r"confined|supported|embedded)\b", link.replace("in situ", "")):
                            continue
                        if source == "abstract" and not (
                                (_SELF_REPORT.search(clause[:match.start()])
                                 and re.search(r"\b(?:report|reported|present|presented|describe|"
                                               r"described|demonstrate|demonstrated|achieve|achieved)\b",
                                               clause[:match.start()]))
                                or re.search(r"\b(?:is|are|was|were|has been|have been) "
                                             r"(?:reported|described|presented)\b", clause[match.end():])):
                            continue
                    elif relation == "active":
                        if source == "title" or not _SELF_REPORT.search(clause[:match.start()]):
                            continue
                        if re.search(r"\bto\s*$", clause[:match.start()]):
                            continue
                        if re.search(r"\b(?:to|for|by|with|on|into|in)\b", link):
                            continue
                    else:
                        # Past participles preceding the entity belong to the
                        # active pattern; here the framework itself is prepared.
                        if re.search(r"\b(?:of|for|by|with|on|into|in|to|as|used|use|"
                                     r"employed|utilized|utilised)\b", link):
                            continue
                        if re.search(r"\b(?:data|results?|measurements?|properties|isotherms?|curves?)\b"
                                     r"[^.;:!?]{0,80}\b(?:about|of|for|on)\s*$",
                                     clause[:match.start("entity")]):
                            continue
                        if source == "abstract" and not re.search(
                                r"\b(?:was|were|is|are|has|have|been)\b", link):
                            continue
                    # A computational title can mention hypothetical synthesis.
                    # Only an explicit experimental abstract claim overrides it.
                    if source == "title" and (
                            primary_result["title_computational_focus"]
                            or primary_result["abstract_computational_focus"]):
                        continue
                    snippet = "{}: {}".format(source, raw)
                    if snippet not in evidence:
                        evidence.append(snippet)
    if denied_preparation and not any(item.startswith("abstract:") for item in evidence):
        # An unqualified title does not overcome an abstract reporting only a
        # failed/negated preparation. Preserve the primary-topic audit instead.
        evidence = []
    return evidence


def classify_topic(title, abstract, document_type=""):
    """Return one topic plus evidence, scores, and explicit review diagnostics.

    Inputs may be strings or spreadsheet blanks. Output contains only standard
    Python scalar values and can be expanded directly into a dataframe.
    """
    title_text = _normalize(title)
    abstract_text = _normalize(abstract)
    is_review_article, review_source, review_evidence = detect_review_article(title, abstract, document_type)
    scores = {}
    title_scores = {}
    evidence_parts = []
    category_evidence = {category: [] for category in CATEGORIES}

    for category in CATEGORIES:
        title_total = 0.0
        abstract_total = 0.0
        for group_name, title_weight, abstract_weight, pattern in _GROUPS[category]:
            for field, text_value in (("title", title_text), ("abstract", abstract_text)):
                # Retain at most three distinct phrases for an audit-friendly
                # evidence column; points still accrue only once per group.
                matches = list(dict.fromkeys(match.group(0) for match in pattern.finditer(text_value)))
                if not matches:
                    continue
                if field == "title":
                    title_total += 4.0 * title_weight
                else:
                    abstract_total += abstract_weight
                entry = "{} [{}; {}]: {}".format(category, field, group_name, ", ".join(matches[:3]))
                evidence_parts.append(entry)
                category_evidence[category].append(entry)
        title_scores[category] = min(20.0, title_total)
        scores[category] = title_scores[category] + min(6.0, abstract_total)

    signals = _contribution_signals(title_text, abstract)
    computational_focus = bool(_COMPUTATIONAL_TITLE.search(title_text))
    purpose_match = _PURPOSE_MARKER.search(title_text)
    application_purpose = bool(purpose_match and
                               _APPLICATION.search(title_text[purpose_match.end():]))
    molecular_reaction_title = any(pattern.search(title_text) for name, _, _, pattern
                                  in _GROUPS[CATEGORIES[0]] if name == "molecular reaction development")
    title_material_catalysis = bool(
        _SOLID_CATALYST.search(title_text)
        and ((_CATALYSIS.search(title_text) and
              (_CATALYTIC_OBJECTIVE.search(title_text) or molecular_reaction_title))
             or (molecular_reaction_title and _FRAMEWORK_REACTION_AGENT.search(title_text))))
    # A named organic reaction performed over a solid/MOF catalyst is an
    # application, even when the title calls the product-making step synthesis.
    # Require catalyst and solid-material evidence in the SAME abstract sentence
    # to avoid linking unrelated background mentions across the abstract.
    abstract_material_catalysis = any(
        _SOLID_CATALYST.search(sentence) and _CATALYSIS.search(sentence)
        and (_CATALYTIC_OBJECTIVE.search(sentence) or _PERFORMANCE.search(sentence))
        and (_SELF_REPORT.search(sentence) or _REPORTED_CATALYST_USE.search(sentence))
        and not re.search(r"\b(?:can|could|may|might|previously|reported by|in the literature)\b", sentence)
        for _, sentence in _abstract_sentences(abstract))
    purpose_text = title_text[purpose_match.end():] if purpose_match else ""
    purpose_is_modification = bool(_POSTSYNTHETIC_MODIFICATION.search(purpose_text))
    external_catalytic_purpose = bool(purpose_text and not purpose_is_modification and (
        _CATALYTIC_OBJECTIVE.search(purpose_text)
        or (_CATALYSIS.search(purpose_text) and _SOLID_CATALYST.search(title_text))
        or any(pattern.search(purpose_text) for name, _, _, pattern in _GROUPS[CATEGORIES[0]]
               if name == "molecular reaction development")))
    postsynthetic_framework_modification = bool(
        _POSTSYNTHETIC_MODIFICATION.search(title_text)
        and _FRAMEWORK_SUBJECT.search(title_text + " " + abstract_text)
        and not application_purpose and not external_catalytic_purpose)
    # Named groups avoid assigning routine hydrothermal conditions the same
    # status as reaction development or post-synthetic transformations.
    synthesis_focus_title = any(
        pattern.search(title_text)
        for name, _, _, pattern in _GROUPS[CATEGORIES[0]]
        if name in ("synthetic routes", "synthesis development",
                    "post-synthetic chemistry", "molecular reaction development"))
    experimental_title = bool(_EXPERIMENTAL_TITLE.search(title_text))
    computation_evidence = computational_focus or bool(signals["computation"])
    mixed_work = bool(computation_evidence and (experimental_title or signals["synthesis"]))
    focus_category = None
    focus_reason = ""

    if is_review_article:
        focus_category = CATEGORIES[1]
        focus_reason = ("User taxonomy: review articles belong to Theory & modeling, "
                        "regardless of the topic reviewed. Review identified from " + review_source + ".")
    elif computation_evidence and not mixed_work:
        focus_category = CATEGORIES[1]
        focus_reason = ("Title explicitly frames a computational/theoretical study."
                        if computational_focus else
                        "Abstract explicitly states a primary computational/theoretical investigation.")
    elif postsynthetic_framework_modification:
        focus_category = CATEGORIES[0]
        focus_reason = ("Title explicitly modifies the framework/linker post-synthetically; "
                        "a catalyst descriptor alone does not establish an external catalytic application.")
    elif signals["title_application"] or title_material_catalysis or (
            molecular_reaction_title and abstract_material_catalysis):
        focus_category = CATEGORIES[3]
        focus_reason = ("A framework or solid catalyst is used for a molecular reaction: catalytic material application."
                        if title_material_catalysis or (molecular_reaction_title and abstract_material_catalysis)
                        else "Title foregrounds an application or material performance objective.")
    elif (synthesis_focus_title and (molecular_reaction_title or _METHOD_DEVELOPMENT.search(title_text))):
        focus_category = CATEGORIES[0]
        focus_reason = "Title foregrounds synthetic-method development or a chemical transformation."
    elif (signals["abstract_method"] and not signals["structural_title"]
          and not signals["abstract_application"]):
        focus_category = CATEGORIES[0]
        focus_reason = "Abstract foregrounds synthetic route development or a post-synthetic transformation."
    elif signals["construction_structure"] and not molecular_reaction_title:
        focus_category = CATEGORIES[2]
        focus_reason = ("Synthesis/construction is coupled to structural characterization or network design; "
                        "brief supporting property measurements do not set the primary topic.")
    elif synthesis_focus_title:
        focus_category = CATEGORIES[0]
        focus_reason = "Title foregrounds a synthetic route, post-synthetic transformation, or molecular reaction."
    elif signals["abstract_method"] and not signals["abstract_application"]:
        focus_category = CATEGORIES[0]
        focus_reason = "Abstract foregrounds synthetic route development or a post-synthetic transformation."
    elif signals["abstract_application"]:
        focus_category = CATEGORIES[3]
        focus_reason = "Abstract links a stated application to performance or uptake results."
    elif computation_evidence:
        focus_category = CATEGORIES[1]
        focus_reason = "Explicit computational contribution in mixed work; no clearer alternative primary focus."

    if focus_category:
        scores[focus_category] += 30.0
        evidence_parts.append("{} [primary contribution rule; +30]: {}".format(focus_category, focus_reason))

    # Sorting is stable: exact ties follow the declared category order.
    ranked = sorted(CATEGORIES, key=lambda category: scores[category], reverse=True)
    winner, runner_up = ranked[:2]
    best_score, next_score = scores[winner], scores[runner_up]
    margin = best_score - next_score
    reasons = []
    fallback_used = False
    fallback_basis = ""
    if best_score == 0:
        if title_text or abstract_text:
            category, fallback_basis, fallback_evidence = _fallback_topic(title_text, abstract_text)
            fallback_used = True
            evidence_parts.append(fallback_evidence)
            focus_reason = "Provisional {} assignment from {}; human validation required.".format(category, fallback_basis)
            reasons.append("Low-confidence fallback: no specific topic evidence; " + fallback_basis)
        else:
            category = UNCLASSIFIED
            reasons.append("No topic-specific phrase evidence")
    else:
        category = winner
        if best_score < 4.0:
            reasons.append("Weak topic evidence (score below 4)")
        if margin < 3.0 or (next_score >= 4.0 and best_score < 1.35 * next_score):
            reasons.append("Competing categories: {} / {}".format(winner, runner_up))
        if title_scores[winner] == 0:
            reasons.append("Assigned category is supported only by abstract evidence")
        if mixed_work:
            reasons.append("Mixed computational and experimental/synthetic contributions")
        if signals["review_article"]:
            reasons.append("Review/perspective article; topic is coverage rather than a single original contribution")
        # Even a numerical winner can mask explicitly overlapping title topics.
        substantial_title_categories = [name for name in CATEGORIES if title_scores[name] >= 10.0]
        if len(substantial_title_categories) > 1:
            reasons.append("Multiple substantial topics in title: " + " / ".join(substantial_title_categories))
        if signals["construction_structure"] and synthesis_focus_title:
            reasons.append("Structural construction and synthetic-method contributions overlap")
        if signals["construction_structure"] and signals["title_application"]:
            reasons.append("Structural construction and application contributions overlap")
        if _POSTSYNTHETIC_MODIFICATION.search(title_text) and _CATALYSIS.search(title_text):
            reasons.append("Post-synthetic modification and catalysis both appear in title; confirm primary contribution")
    if is_review_article:
        # Topic overlap is expected for a review and does not change this rule.
        reasons = ([] if review_source == "Document Type" else
                   ["Review-article genre inferred from title/abstract; confirm if needed"])
    if not title_text:
        reasons.append("Missing title")
    if title_text and not abstract_text:
        reasons.append("Missing abstract; contribution inferred from title only")
    if not title_text and not abstract_text:
        reasons.append("Missing title and abstract")

    result = {
        "category": category,
        "review_needed": bool(reasons),
        "low_confidence": bool(reasons),
        "fallback_used": fallback_used,
        "fallback_basis": fallback_basis,
        "review_reason": "; ".join(reasons),
        "evidence": " | ".join(evidence_parts),
        "runner_up": runner_up if next_score > 0 else "",
        "score_margin": round(margin, 2),
        "title_computational_focus": computational_focus,
        "title_application_purpose": application_purpose,
        "primary_topic_reason": focus_reason or (
            "No specific topic evidence." if category == UNCLASSIFIED else
            "Phrase-group evidence favors {}; no explicit primary-contribution rule applies.".format(category)),
        "abstract_computational_focus": bool(signals["computation"]),
        "primary_computation_evidence": " | ".join(signals["computation"][:3]),
        "supporting_computation_evidence": " | ".join(signals["supporting_computation"][:3]),
        "has_experimental_synthesis": bool(signals["synthesis"]),
        "experimental_synthesis_evidence": " | ".join(signals["synthesis"][:3]),
        "structural_construction_evidence": " | ".join(signals["structure"][:3]),
        "mixed_experimental_computational": mixed_work,
        "is_review_or_perspective": signals["review_article"] or is_review_article,
        "is_review_article": is_review_article,
        "review_policy_applied": is_review_article,
        "review_detection_source": review_source,
        "review_detection_evidence": review_evidence,
        "classifier_version": CLASSIFIER_VERSION,
    }
    for name, key in zip(CATEGORIES, _SCORE_KEYS):
        result[key] = round(scores[name], 2)
    # Keep all v4 diagnostics untouched. The public category is a separate,
    # explicitly documented grouping; its counts must never feed this rule.
    result["primary_topic_category"] = result["category"]
    result["primary_topic_evidence"] = result["evidence"]
    framework_evidence = _framework_preparation_evidence(title, abstract, result)
    result["has_framework_synthesis"] = bool(framework_evidence)
    result["framework_synthesis_evidence"] = " | ".join(framework_evidence)
    result["taxonomy_policy"] = TAXONOMY_POLICY
    result["taxonomy_reassigned"] = bool(framework_evidence and category != CATEGORIES[0])
    result["reassignment_reason"] = ""
    result["classification_reason"] = result["primary_topic_reason"]
    if framework_evidence:
        result["category"] = CATEGORIES[0]
        result["classification_reason"] = (
            "Synthesis-inclusive taxonomy: this original paper explicitly reports "
            "preparation of a MOF/coordination framework, network, or polymer; "
            "routine preparation takes precedence over structural or application focus.")
        if result["taxonomy_reassigned"]:
            result["reassignment_reason"] = (
                "Grouped as Chemical synthesis by the synthesis-inclusive policy; "
                "the retained v4 primary topic is {}.".format(category))
    return result


def _run_sanity_checks():
    """Critical topic distinctions and invariants, not an accuracy benchmark."""
    cases = [
        ("Mechanochemical synthesis of metal organic frameworks", "", CATEGORIES[0]),
        ("Post-synthetic modification of zirconium frameworks", "", CATEGORIES[0]),
        ("Topological design of interpenetrated coordination networks", "", CATEGORIES[2]),
        ("Crystal packing and polymorphism in coordination networks", "", CATEGORIES[2]),
        ("Facile synthesis of a metal organic framework for hydrogen storage", "", CATEGORIES[3]),
        ("Density functional theory study of carbon dioxide adsorption in a metal organic framework", "", CATEGORIES[1]),
        ("Machine learning for gas adsorption and separation in MOFs", "", CATEGORIES[1]),
        ("A fluorescent framework for sensing aqueous pollutants", "DFT calculations explain the observed response.", CATEGORIES[3]),
        ("C-H functionalization by a porous framework", "", CATEGORIES[3]),
        ("Photocatalytic hydrogen production by a porous framework", "", CATEGORIES[3]),
        ("New metal organic frameworks", "Materials were characterized by single-crystal X-ray diffraction.", CATEGORIES[2]),
        ("A new material and model", "", CATEGORIES[2]),
        (None, float("nan"), UNCLASSIFIED),
    ]
    for title, abstract, expected in cases:
        result = classify_topic(title, abstract)
        assert result["primary_topic_category"] == expected, (title, expected, result)

    # Repeated abstract language cannot multiply a phrase-group score.
    once = classify_topic("New framework", "Adsorption was studied.")
    repeated = classify_topic("New framework", "Adsorption was studied. " * 30)
    assert once["score_functional_materials"] == repeated["score_functional_materials"]
    assert classify_topic("", "") ["review_needed"]
    assert classify_topic("Crystal growth and solvothermal synthesis", "")["review_needed"]
    assert classify_topic("Experimental and computational study of adsorption", "")["review_needed"]
    print("Passed {} classification cases and 4 review/invariance checks.".format(len(cases)))

In [ ]:
# Classify the before set once, then reuse its categories for matched selected papers.
before_papers = unique_papers(before_rows)
unmatched_after = after_rows.loc[~after_rows.matched_to_before.astype(bool)].copy()
paper_pool = pd.concat([before_papers, unique_papers(unmatched_after)], ignore_index=True)
paper_pool = paper_pool.drop_duplicates('paper_id').reset_index(drop=True)
annotations = []
for i, (title, abstract, document_type) in enumerate(zip(paper_pool['Article Title'], paper_pool['Abstract'], paper_pool['Document Type']), start=1):
    annotations.append(classify_topic(title, abstract, document_type))
    if i % 3000 == 0:
        print('Classified {:,} / {:,} unique papers'.format(i, len(paper_pool)))
classified = pd.concat([paper_pool, pd.DataFrame(annotations)], axis=1)
classified = apply_overrides(classified, OVERRIDE_PATH, CATEGORIES)

before_classified = attach_classification(before_rows, classified)
after_classified = attach_classification(after_rows, classified)
after_y_classified = after_classified.loc[after_classified.selected_Y].copy()
assert before_classified.category.notna().all()
assert after_classified.category.notna().all()
assert classified.paper_id.is_unique
assert set(classified.category).issubset(set(CATEGORIES) | {'Unclassified'})
print('Classified {:,} distinct papers; {:,} currently flagged for review.'.format(
    len(classified), classified.review_needed.sum()))
full_paper_decisions = attach_full_decisions(
    before_classified.drop_duplicates('paper_id'), decision_rows, DECISION_COLUMN)
explicit_y_ids = set(full_paper_decisions.loc[full_paper_decisions.triage_decision.eq('Y'), 'paper_id'])
selected_y_ids = set(after_y_classified.paper_id)
if explicit_y_ids != selected_y_ids:
    raise ValueError('The full decision workbook Y set differs from SELECTED 7000 / LABEL_COLUMN. '
                     'Check that both settings refer to the same triage model.')
print('Verified: explicit full-corpus Y decisions equal the selected Y paper set.')
review_articles = full_paper_decisions.loc[full_paper_decisions.is_review_article].copy()
print('Review articles assigned to Theory & modeling:', len(review_articles))
display(review_articles.triage_decision.value_counts().rename('Review articles').to_frame())

## 4. Before triage, retained (Y), and excluded (N)

**Y and N are separate groups.** The earlier before-triage bars included retained papers;
they were not N bars. The new chart uses explicit decisions, with no category or decision
changed to produce a desired bar pattern. Unknown or conflicting decisions are reported
separately and never silently treated as N.

The summary counts **unique papers** in all three columns. **Before = Y + N** when
there are no Unknown or Conflict decisions. N comes from the explicit full decision
workbook; it is never inferred from absence in the selected workbook. Review columns
show how many rough classifications still need inspection. The row table projects
canonical decisions onto the original full-corpus rows to reconcile with Excel.

In [ ]:
yn_counts = count_yn(full_paper_decisions, CATEGORIES)
print('UNIQUE PAPERS: explicit Y/N decisions')
display(yn_counts)
primary_topic_decisions = full_paper_decisions.copy()
primary_topic_decisions['category'] = primary_topic_decisions.primary_topic_category
primary_topic_counts = count_yn(primary_topic_decisions, CATEGORIES)
taxonomy_transitions = pd.crosstab(full_paper_decisions.primary_topic_category,
                                  full_paper_decisions.category).reindex(
    index=list(CATEGORIES), columns=list(CATEGORIES), fill_value=0)
taxonomy_transitions.index.name = 'Original primary topic'
taxonomy_reassignments = full_paper_decisions.loc[full_paper_decisions.taxonomy_reassigned,
    ['paper_id', 'DOI', 'Article Title', 'primary_topic_category', 'category',
     'framework_synthesis_evidence', 'reassignment_reason', 'triage_decision',
     'review_needed', 'review_reason']].copy()
print('Papers included in Chemical synthesis by explicit framework-preparation evidence:',
      len(taxonomy_reassignments))
print('Original primary topics are preserved in the audit and primary_topic_counts.csv.')
synthesis_summary = full_paper_decisions.groupby('category').agg(
    Papers=('paper_id', 'size'),
    Papers_with_synthesis_evidence=('has_experimental_synthesis', 'sum'))
synthesis_summary = synthesis_summary.reindex(list(CATEGORIES) + ['Unclassified'], fill_value=0)
synthesis_summary['Synthesis evidence (%)'] = (
    100 * synthesis_summary.Papers_with_synthesis_evidence / synthesis_summary.Papers.replace(0, float('nan'))).round(1)
synthesis_summary.index.name = 'Category'
print('INDEPENDENT METHOD TAG: experimental synthesis evidence in title/abstract')
display(synthesis_summary)

category_changes = pd.DataFrame(columns=['paper_id', 'category_v1', 'category_current'])
method_comparison = pd.DataFrame()
if PRIOR_TOPICS_PATH.exists():
    prior = pd.read_csv(PRIOR_TOPICS_PATH, dtype=str, keep_default_na=False)
    if not prior.paper_id.is_unique:
        raise ValueError('Prior category snapshot must have unique paper IDs.')
    category_changes = prior[['paper_id', 'category']].rename(columns={'category': 'category_v1'}).merge(
        classified[['paper_id', 'Article Title', 'category', 'primary_topic_category', 'classification_reason', 'review_needed']],
        on='paper_id', how='inner', validate='one_to_one').rename(columns={'category': 'category_current'})
    method_comparison = pd.crosstab(category_changes.category_v1, category_changes.category_current,
                                    margins=True, margins_name='TOTAL')
    category_changes = category_changes.loc[category_changes.category_v1.ne(category_changes.category_current)]
    print('Papers reassigned by the revised method:', len(category_changes))
    display(method_comparison)

is_subset = bool(after_y_classified.matched_to_before.all())
counts_by_category = count_categories(before_classified, after_y_classified, CATEGORIES,
                                     unique=True, subset=is_subset, full_decisions=full_paper_decisions)
row_counts_by_category = count_categories(before_classified, after_y_classified, CATEGORIES,
                                         unique=False, subset=False, full_decisions=full_paper_decisions)
print('UNIQUE PAPERS — after column:', LABEL_COLUMN)
display(counts_by_category[['Before triage', 'After triage (Y)', 'After triage (N)']])
display(counts_by_category)
print('SPREADSHEET ROWS — includes duplicate records')
display(row_counts_by_category)
if not is_subset:
    print('Retention percentages omitted: some Y papers did not safely match the before corpus.')

# Sensitivity table: all actual decision columns, with the same fixed topic assignments.
label_comparison = []
for col in TRIAGE_COLUMNS:
    if col in after_classified:
        selected = after_classified.loc[is_y(after_classified[col])]
        table = count_categories(before_classified, selected, CATEGORIES, unique=True,
                                 subset=bool(selected.matched_to_before.all()))
        label_comparison.append(table[['After triage (Y)']].rename(columns={'After triage (Y)': col + ' Y'}))
label_comparison = pd.concat(label_comparison, axis=1)
print('UNIQUE Y PAPERS BY DECISION COLUMN — blanks are excluded')
display(label_comparison)

In [ ]:
plot_counts = yn_counts.loc[list(CATEGORIES)]
plt.rcParams.update({'font.size': 11, 'axes.spines.top': False, 'axes.spines.right': False})
fig, ax = plt.subplots(figsize=(10.8, 5.8))
x = list(range(len(plot_counts)))
width = 0.30
b1 = ax.barh([v - width/2 for v in x], plot_counts['N'], height=width,
            label='N', color='#efbd6d', edgecolor='#d2a45b', linewidth=0.5)
b2 = ax.barh([v + width/2 for v in x], plot_counts['Y'], height=width,
            label='Y', color='#8fbd83', edgecolor='#76a66a', linewidth=0.5)
for bars in [b1, b2]:
    ax.bar_label(bars, labels=['{:,}'.format(int(b.get_width())) for b in bars], padding=5, fontsize=10)
ax.set_yticks(x)
ax.set_yticklabels(plot_counts.index)
ax.invert_yaxis()
ax.set_xlabel('Unique papers')
ax.set_title('Literature triage by synthesis-inclusive category', loc='left', weight='bold', pad=18)
ax.set_xlim(0, max(1, plot_counts[['Y', 'N']].to_numpy().max()) * 1.14)
ax.legend(frameon=False, ncol=2, loc='upper right')
ax.xaxis.grid(True, alpha=0.18)
ax.set_axisbelow(True)
unclassified = yn_counts.loc['Unclassified']
unresolved = int(yn_counts.loc['TOTAL', ['Unknown', 'Conflict']].sum())
fig.text(0.02, 0.065, 'Actual decisions: Full_gpt-5.xlsx / Agent_YN. Topic rules v' + CLASSIFIER_VERSION + '; review flags included.',
         fontsize=8.5, color='#52606d')
fig.text(0.02, 0.025, 'Outside the four bars: unclassified Y={:,}, N={:,}; unknown/conflicting decisions={:,}.'.format(
    int(unclassified.Y), int(unclassified.N), unresolved), fontsize=8.5, color='#52606d')
fig.tight_layout(rect=[0, 0.10, 1, 1])
fig.savefig(OUTPUT_DIR / 'triage_Y_N_by_topic.png', dpi=240, bbox_inches='tight')
fig.savefig(OUTPUT_DIR / 'triage_Y_N_by_topic.pdf', bbox_inches='tight')
# Keep the previous chart filenames as aliases for the revised main plot.
fig.savefig(OUTPUT_DIR / 'before_after_topics.png', dpi=240, bbox_inches='tight')
fig.savefig(OUTPUT_DIR / 'before_after_topics.pdf', bbox_inches='tight')
plt.show()

## 5. Inspect individual papers and correct categories

The next cell displays a reproducible sample from every category and the first flagged
papers. Use the exported audit table to inspect **every paper**, including the title,
abstract, category scores, matched topic evidence, and review reason.

To correct classifications, copy any rows from `outputs/review_queue.csv` (or
`outputs/all_unique_papers.csv`) into `category_overrides.csv` beside the notebook.
Keep `paper_id`; fill a `manual_category` column with an exact category name from the
rubric, and optionally add `review_note`. Extra columns are accepted; blank edits are
ignored. A manual category marks that paper reviewed. Re-run from Section 3 to update
both before and after counts together. Do not edit `automatic_category` or the raw files.

In [ ]:
inspection_columns = ['paper_id', 'Article Title', 'category', 'primary_topic_category',
                      'classification_reason', 'framework_synthesis_evidence',
                      'review_needed', 'review_reason', 'evidence']
sample = pd.concat([group.sample(n=min(3, len(group)), random_state=42)
                    for _, group in classified.groupby('category')])
display(sample[inspection_columns])
review_queue = classified.loc[classified.review_needed].copy()
review_queue['manual_category'] = ''
review_queue = review_queue.sort_values(['category', 'paper_id'])
print('Review queue:', len(review_queue), 'unique papers')
display(review_queue[inspection_columns].head(15))

# Change PAPER_QUERY to part of a title or DOI to inspect particular papers.
PAPER_QUERY = ''
if PAPER_QUERY:
    matches = classified['Article Title'].str.contains(PAPER_QUERY, case=False, regex=False) | classified['DOI'].str.contains(PAPER_QUERY, case=False, regex=False)
    display(classified.loc[matches, inspection_columns + ['Abstract']])

## 6. Export results and verify totals

`all_unique_papers.csv` and `paper_topic_audit.xlsx` contain the complete classifications.
`doi_classification.csv` is the compact two-column DOI/classification list. The companion
`doi_classification_triage.csv` adds the existing Y/N decision and review flag.
`resolved_unclassified.csv` traces every previously unclassified paper to its new
category, matching evidence, and review reason.
`taxonomy_reassignments.csv` records the synthesis-inclusive reassignments and exact
preparation evidence. `primary_topic_counts.csv` preserves the previous main-topic
counts, and `taxonomy_transition_counts.csv` reconciles the two category definitions.
`before_classified.csv`, `after_all_classified.csv`, and `after_Y_classified.csv` preserve
source rows. The Excel audit also includes the counts, label comparison, duplicate
records, review queue, unmatched Y records, and conflicting decisions. The manifest
records input hashes, chosen label, row and paper counts, and the classification method.
No workbook content is sent to an external service.

In [ ]:
duplicate_rows = pd.concat([
    before_classified.loc[before_classified.paper_id.duplicated(keep=False)].assign(dataset='before'),
    after_classified.loc[after_classified.paper_id.duplicated(keep=False)].assign(dataset='selected')
], ignore_index=True)
before_ids = set(before_rows.paper_id)
after_y_ids = set(after_y_rows.paper_id)
classified['in_before'] = classified.paper_id.isin(before_ids)
classified['after_selected_Y'] = classified.paper_id.isin(after_y_ids)
classified['triage_decision'] = classified.paper_id.map(full_paper_decisions.set_index('paper_id').triage_decision).fillna('Unknown')

doi_classification = full_paper_decisions[['doi_key', 'category']].rename(
    columns={'doi_key': 'DOI', 'category': 'Classification'}).sort_values('DOI')
if doi_classification.DOI.eq('').any() or doi_classification.DOI.duplicated().any():
    raise ValueError('The compact DOI export requires one nonempty DOI per paper.')
doi_classification_triage = full_paper_decisions[
    ['doi_key', 'category', 'triage_decision', 'review_needed']].rename(columns={
        'doi_key': 'DOI', 'category': 'Classification', 'triage_decision': 'Triage decision',
        'review_needed': 'Needs review'}).sort_values('DOI')
resolved_unclassified = pd.DataFrame()
if PREVIOUS_TOPICS_PATH.exists():
    previous = pd.read_csv(PREVIOUS_TOPICS_PATH, dtype=str, keep_default_na=False)
    previous_ids = set(previous.loc[previous.category.eq('Unclassified'), 'paper_id'])
    resolved_unclassified = full_paper_decisions.loc[
        full_paper_decisions.paper_id.isin(previous_ids),
        ['paper_id', 'DOI', 'Article Title', 'category', 'triage_decision',
         'primary_topic_category', 'classification_reason', 'review_needed', 'review_reason', 'evidence']].copy()
    resolved_unclassified['previous_category'] = 'Unclassified'
    print('Previously unclassified papers assigned a topic:',
          int(resolved_unclassified.category.ne('Unclassified').sum()), '/', len(previous_ids))

if PRIMARY_TOPICS_PATH.exists():
    prior_primary = pd.read_csv(PRIMARY_TOPICS_PATH, dtype=str, keep_default_na=False).set_index('paper_id')
    current = full_paper_decisions.set_index('paper_id')
    assert prior_primary.index.is_unique and set(prior_primary.index) == set(current.index)
    current = current.loc[prior_primary.index]
    assert current.primary_topic_category.eq(prior_primary.category).all(), 'Original primary topic changed'
    assert current.triage_decision.eq(prior_primary.triage_decision).all(), 'Saved triage decision changed'
    print('Verified: every original primary topic and saved Y/N decision is preserved.')

tables = {
    'primary_topic_counts': primary_topic_counts.reset_index(),
    'taxonomy_transition_counts': taxonomy_transitions.reset_index(),
    'taxonomy_reassignments': taxonomy_reassignments,
    'doi_classification': doi_classification,
    'doi_classification_triage': doi_classification_triage,
    'resolved_unclassified': resolved_unclassified,
    'Y_N_counts_by_topic': yn_counts.reset_index(),
    'synthesis_evidence_by_topic': synthesis_summary.reset_index(),
    'category_changes_since_v1': category_changes,
    'method_transition_counts': method_comparison.reset_index(),
    'full_decision_audit': full_paper_decisions,
    'review_articles': review_articles,
    'counts_by_category': counts_by_category.reset_index(),
    'row_counts_by_category': row_counts_by_category.reset_index(),
    'label_comparison': label_comparison.reset_index(),
    'coverage': coverage,
    'label_inventory': label_inventory(after_rows),
    'all_unique_papers': classified,
    'before_classified': before_classified,
    'after_all_classified': after_classified,
    'after_Y_classified': after_y_classified,
    'review_queue': review_queue,
    'duplicate_records': duplicate_rows,
    'unmatched_Y': unmatched_y,
    'conflicting_decisions': label_conflicts,
}
for name, table in tables.items():
    table.to_csv(OUTPUT_DIR / (name + '.csv'), index=False, encoding='utf-8-sig')

# The compact workbook carries complete unique-paper results and selected-row results.
workbook_tables = {k: v for k, v in tables.items() if k not in ['before_classified', 'after_all_classified']}
# Stream rows to disk: large abstract/evidence tables need not live as Excel cells in memory.
from openpyxl import Workbook
from openpyxl.cell import WriteOnlyCell
from openpyxl.utils import get_column_letter
workbook = Workbook(write_only=True)
for name, table in workbook_tables.items():
    sheet = workbook.create_sheet(title=name[:31])
    sheet.freeze_panes = 'A2'
    sheet.auto_filter.ref = 'A1:{}{}'.format(get_column_letter(len(table.columns)), len(table) + 1)
    sheet.append(list(table.columns))
    for values in table.itertuples(index=False, name=None):
        excel_row = []
        for value in values:
            value = None if pd.isna(value) else value
            cell = WriteOnlyCell(sheet, value=value)
            if isinstance(value, str):
                cell.data_type = 's'
            excel_row.append(cell)
        sheet.append(excel_row)
workbook.save(OUTPUT_DIR / 'paper_topic_audit.xlsx')

manifest = {
    'run_time_utc': datetime.now(timezone.utc).isoformat(),
    'inputs': {str(p): hashlib.sha256(p.read_bytes()).hexdigest() for p in [BEFORE_PATH, AFTER_PATH, DECISION_PATH]},
    'sheet': SHEET_NAME,
    'label_column': LABEL_COLUMN,
    'YN_source': str(DECISION_PATH),
    'YN_column': DECISION_COLUMN,
    'explicit_Y_unique': int(yn_counts.loc['TOTAL', 'Y']),
    'explicit_N_unique': int(yn_counts.loc['TOTAL', 'N']),
    'unresolved_decisions': int(yn_counts.loc['TOTAL', ['Unknown', 'Conflict']].sum()),
    'classifier_uses_triage_decisions': False,
    'review_article_policy': 'All identified review articles are assigned to Theory & modeling',
    'identified_review_articles': len(review_articles),
    'review_article_detection': 'Document Type, or explicit title/abstract self-description; evidence exported',
    'main_chart': 'Disjoint explicit Y (green) and N (orange); not before-versus-after',
    'label_rule': 'stripped case-insensitive exact Y; any Y per duplicate paper ID',
    'classification': 'Rough deterministic synthesis-inclusive grouping; preserved original main topic plus explicit experimental framework preparation; not generated by an LLM',
    'taxonomy_policy': 'synthesis-inclusive-v1',
    'taxonomy_reassigned_unique': int(classified.taxonomy_reassigned.sum()),
    'framework_synthesis_evidence_unique': int(classified.has_framework_synthesis.sum()),
    'primary_topic_preserved': True,
    'classification_generated_by_llm': False,
    'classifier_version': CLASSIFIER_VERSION,
    'classification_validated_against_expert_labels': False,
    'unique_identity': 'normalized DOI, else WOS ID, else normalized title, else source row',
    'before_rows': len(before_rows),
    'before_unique_papers': before_rows.paper_id.nunique(),
    'after_Y_rows': len(after_y_rows),
    'after_Y_unique_papers': after_y_rows.paper_id.nunique(),
    'unmatched_Y_rows': len(unmatched_y),
    'unique_papers_needing_review': int(classified.review_needed.sum()),
    'fallback_assignments': int(classified.fallback_used.sum()),
    'fallback_basis_counts': classified.loc[classified.fallback_used, 'fallback_basis'].value_counts().to_dict(),
    'unclassified_unique_papers': int(full_paper_decisions.category.eq('Unclassified').sum()),
    'previously_unclassified_papers': len(resolved_unclassified),
    'previously_unclassified_resolved': int(resolved_unclassified.category.ne('Unclassified').sum()) if len(resolved_unclassified) else 0,
    'partition_check': 'Before = explicit Y + explicit N + Unknown + Conflict, at DOI and source-row levels',
    'manual_overrides_applied': int(classified.manual_reviewed.sum()),
    'override_sha256': hashlib.sha256(OVERRIDE_PATH.read_bytes()).hexdigest() if OVERRIDE_PATH.exists() else None,
    'python': platform.python_version(),
    'pandas': pd.__version__,
}
(OUTPUT_DIR / 'run_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
assert counts_by_category.loc['TOTAL', 'Before triage'] == before_rows.paper_id.nunique()
assert counts_by_category.loc['TOTAL', 'After triage (Y)'] == after_y_rows.paper_id.nunique()
assert row_counts_by_category.loc['TOTAL', 'Before triage'] == len(before_rows)
assert row_counts_by_category.loc['TOTAL', 'After triage (Y)'] == len(after_y_rows)
print('Verified totals. Results saved to:', OUTPUT_DIR.resolve())
display(counts_by_category[['Before triage', 'After triage (Y)', 'After triage (N)']])
print('Compact DOI classification list (full list is saved as doi_classification.csv):')
display(doi_classification.head(12))

## 7. Compare four topic definitions and check the requested Y:N ratios

Each definition below assigns a topic from the same title, abstract, and document type.
The alternatives change the meaning of a category; they are not validated improvements
in classification accuracy. All identified reviews remain **Theory & modeling**.
Every paper is included under every definition, including **Unclassified** papers.
Existing manual category corrections, if present, are retained across definitions.

The requested checks are **Chemical synthesis >= 4:1**, **Crystal engineering > 6:1**
(with a separate **> 7:1** check), and **Functional materials Y > N**.
These checks summarize outcomes after classification; they do not determine topic
assignments or select a preferred definition. Moving an N paper requires the same
positive topic evidence that would move a Y paper. No additional rule is fitted to
make the displayed ratios pass.

The four criteria were specified before calculating their comparison results. They
are exploratory choices made with knowledge of the requested targets, not a
preregistered or independently validated classification study. Expert review of a
sample blinded to triage labels is needed to decide which definition best matches
the intended scientific categories.

In [ ]:
"""Fixed, cumulative topic-definition sensitivity analysis (standard library only).

These criteria change the meaning of category boundaries, not triage decisions.
Only bibliographic text and the original text-derived category are inputs. The
criteria were specified without inspecting decision labels or category totals.
Evidence scores are not accuracy estimates; disputed/missing evidence is flagged.
"""

import re



CRITERIA = (
    "baseline_current", "framework_scope", "property_priority", "computation_inclusive",
)
CRITERION_LABELS = {
    "baseline_current": "Current synthesis-inclusive baseline (v5)",
    "framework_scope": "Framework synthesis scope",
    "property_priority": "Property priority + strict synthesis/structure",
    "computation_inclusive": "Include primary computation in mixed studies",
}
CRITERION_DESCRIPTIONS = {
    "baseline_current": (
        "Current v5 synthesis-inclusive grouping: original affirmative preparation "
        "claims for MOFs/coordination frameworks, networks, or polymers place papers "
        "in Chemical synthesis, including routine preparation within structure and "
        "application studies. Existing primary-topic synthetic-method, molecular, "
        "and post-synthetic assignments are also retained. Reviews belong to Theory "
        "& modeling under the requested taxonomy. The independent primary-topic "
        "audit is preserved in the baseline outputs."
    ),
    "framework_scope": (
        "Starting from the current v5 grouping, distinguish making/modifying a framework from using it "
        "as a catalyst. Explicit framework-catalyzed molecular transformations or "
        "catalyst-performance contributions become Functional materials unless "
        "a framework-making/modification method is foregrounded. Reviews and "
        "primary theoretical studies remain Theory & modeling."
    ),
    "property_priority": (
        "Starting from framework scope, a functional/property endpoint foregrounded "
        "in the title or stated as a substantive study result takes priority over "
        "experimental synthesis/structure. Luminescence and magnetism in the title "
        "count here. Crystal engineering requires original construction together "
        "with new-structure or structural-design evidence; routine XRD alone is "
        "insufficient. Chemical synthesis requires an explicit preparative method "
        "or chemical-modification contribution. Unsupported assignments become "
        "Unclassified for review, rather than being inferred as theoretical. "
        "Explicit non-framework synthetic-method contributions are retained but flagged."
    ),
    "computation_inclusive": (
        "Starting from property priority, an explicit primary computational method "
        "or contribution in the title/abstract takes priority even in mixed "
        "experimental/computational studies. Supporting-only calculations do not "
        "qualify. Reviews always belong to Theory & modeling."
    ),
}

_CRITERIA_CATALYST = _compile([
    r"catalysts?", r"cataly[sz]\w*", r"catalytic\w*", r"catalysis",
])
_CRITERIA_REACTION = _compile([
    r"(?:c h|c c|c n|c o) (?:bond )?(?:activation|functionalization|functionalisation|formation|coupling)",
    r"(?:suzuki|heck|sonogashira|buchwald hartwig|knoevenagel|hantzsch|biginelli|aldol|pechmann)\w*",
    r"cross coupling", r"cycloaddition", r"esterification", r"transesterification",
    r"amidation", r"acetalization", r"acetalisation", r"alkylation", r"arylation",
    r"hydrogenation", r"oxidation", r"cyclic carbonates?", r"organic transformations?",
    r"organic reactions?", r"substrates?", r"catalytic (?:performance|activity)",
])
_CRITERIA_CATALYTIC_RESULT = _compile([
    r"selectivit\w*", r"conversion\w*", r"yield\w*", r"turnover\w*",
    r"recyclab\w*", r"reusab\w*", r"catalytic (?:performance|activity)",
])
_CRITERIA_OWN_RESULT = _compile([
    r"show\w*", r"exhibit\w*", r"demonstrat\w*", r"achiev\w*",
    r"afford\w*", r"deliver\w*", r"display\w*", r"found", r"measur\w*",
    r"observ\w*", r"investigat\w*", r"evaluat\w*", r"studied",
])
_CRITERIA_BACKGROUND = _compile([
    r"previous\w*", r"reported by", r"in the literature",
    r"could be", r"can be", r"may be", r"might be", r"potential applications?",
])
_CRITERIA_PROPERTY = _compile([
    r"luminescen\w*", r"photoluminescen\w*", r"fluorescen\w*", r"phosphorescen\w*",
    r"magnetic propert\w*", r"magnetism", r"magnetocaloric\w*", r"magnetic behavior",
    r"magnetic behaviour", r"spin crossover", r"ferroelectric\w*", r"dielectric\w*",
    r"photophysic\w*", r"optical propert\w*", r"mechanical propert\w*",
    r"electronic propert\w*", r"thermal conduct\w*", r"elastic propert\w*",
])
_CRITERIA_PROPERTY_RESULT = _compile([
    r"quantum yield", r"lifetime\w*", r"emission (?:wavelength|intensity)",
    r"magnetic susceptibility", r"magnetization", r"magnetisation", r"curie temperature",
    r"transition temperature", r"conductivit\w*", r"modulus", r"elastic constant\w*",
])
_CRITERIA_STRUCTURAL_DESIGN = _compile([
    r"topolog\w*", r"reticular\w*", r"isoreticular\w*", r"crystal engineering",
    r"crystal packing", r"supramolecular (?:assembl\w*|architectur\w*|synthons?)",
    r"interpenetrat\w*", r"polymorph\w*", r"structural diversit\w*",
    r"(?:structure|structural) directing", r"network connectivity",
    r"crystal growth", r"single crystal to single crystal", r"single crystal transformation\w*",
])
_CRITERIA_NEW_STRUCTURES = _compile([
    r"(?:new|novel|previously unreported) (?:\w+ ){0,10}(?:coordination polymer\w*|framework\w*|coordination network\w*|complex\w*|co\s?crystal\w*)",
])
_CRITERIA_CREATION_TITLE = _compile([
    r"synthes(?:is|es)", r"construction", r"self assembl\w*", r"assembly",
    r"crystal growth", r"single crystal transformation\w*",
])
_CRITERIA_MODIFICATION = _compile([
    r"post\s?synthetic (?:modification\w*|exchange\w*|transformation\w*|functionalization\w*|functionalisation\w*)",
    r"(?:linker|ligand) exchange", r"transmetalation\w*", r"transmetallation\w*",
])
_CRITERIA_SUPPORTING_TITLE = re.compile(
    r"\b(?:supported|corroborated|confirmed|rationalized|rationalised) "
    r"(?:by|using|with) (?:\w+ ){0,4}(?:dft|density functional|calculations?|simulations?)\b|"
    r"\b(?:supporting|auxiliary) (?:\w+ ){0,3}(?:dft|calculations?|simulations?)\b"
)


def _criteria_snippets(items):
    """Keep quoted source evidence compact and deterministic."""
    return " | ".join(str(item).strip() for item in items[:3] if str(item).strip())


def _criteria_framework_target(title_text):
    """Distinguish synthesis OF a framework from organic synthesis USING one."""
    match = re.search(r"\b(?:synthes(?:is|es)|preparation|fabrication|construction|growth) of (.+)", title_text)
    if not match:
        return False
    target = re.split(r"\b(?:for|using|over|by|via|with|in|on|employing|cataly[sz]\w*)\b", match.group(1), maxsplit=1)[0]
    return bool(_FRAMEWORK_SUBJECT.search(target))


def assign_topic_criteria(title, abstract, document_type="", baseline_category=None):
    """Return four topic assignments from text; no triage outcomes are accepted.

    ``baseline_category`` may reuse the current text-only v5 result. Manual
    overrides should be applied consistently by the caller after this function.
    """
    title_text = _normalize(title)
    abstract_text = _normalize(abstract)
    full_text = title_text + " " + abstract_text
    base = classify_topic(title, abstract, document_type) if baseline_category is None else None
    base_category = base["category"] if base is not None else baseline_category
    if base_category not in CATEGORIES + (UNCLASSIFIED,):
        raise ValueError("baseline_category must be an existing topic category")
    signals = _contribution_signals(title_text, abstract)
    sentences = _abstract_sentences(abstract)
    review, review_source, review_evidence = detect_review_article(title, abstract, document_type)
    if review:
        return [{
            "criterion": criterion, "category": CATEGORIES[1],
            "reason": "Review article belongs to Theory & modeling under the requested taxonomy.",
            "evidence": "{}: {}".format(review_source, review_evidence),
            "review_needed": review_source != "Document Type",
        } for criterion in CRITERIA]

    missing_text = not title_text or not abstract_text
    base_review = bool(base["review_needed"]) if base else (missing_text or base_category == UNCLASSIFIED)
    assignments = [{
        "criterion": CRITERIA[0], "category": base_category,
        "reason": base["primary_topic_reason"] if base else "Current text-derived v5 synthesis-inclusive assignment.",
        "evidence": base["evidence"] if base else "Current v5 evidence is retained in the paper audit.",
        "review_needed": base_review,
    }]

    framework = bool(_FRAMEWORK_SUBJECT.search(full_text))
    title_route = any(pattern.search(title_text) for name, _, _, pattern in _GROUPS[CATEGORIES[0]]
                      if name in ("synthetic routes", "synthesis development", "post-synthetic chemistry"))
    title_molecular = any(pattern.search(title_text) for name, _, _, pattern in _GROUPS[CATEGORIES[0]]
                          if name == "molecular reaction development")
    modification_title = bool(_CRITERIA_MODIFICATION.search(title_text))
    framework_target = _criteria_framework_target(title_text)
    framework_method_focus = framework and (
        modification_title or (title_route and (framework_target or not title_molecular))
        or (signals["abstract_method"] and not title_molecular))

    own_catalysis = [raw for raw, sentence in sentences
                    if _CRITERIA_CATALYST.search(sentence)
                    and (_CRITERIA_REACTION.search(sentence) or _CRITERIA_CATALYTIC_RESULT.search(sentence))
                    and (_SELF_REPORT.search(sentence) or _CRITERIA_OWN_RESULT.search(sentence))
                    and not _CRITERIA_BACKGROUND.search(sentence)]
    title_catalysis = bool(_CRITERIA_CATALYST.search(title_text)
                          and (_CRITERIA_REACTION.search(title_text)
                               or _CRITERIA_CATALYTIC_RESULT.search(title_text)))
    catalyst_application = framework and (title_catalysis or bool(own_catalysis))
    current = dict(assignments[-1], criterion=CRITERIA[1])
    if base_category != CATEGORIES[1] and catalyst_application and not framework_method_focus:
        current.update(
            category=CATEGORIES[3],
            reason="Framework is used as a catalyst for a molecular transformation/performance study; no foregrounded framework-making method.",
            evidence=_criteria_snippets((["Title: " + str(title)] if title_catalysis else []) + own_catalysis),
            review_needed=missing_text or bool(signals["construction_structure"]),
        )
    assignments.append(current)

    title_property = bool(_APPLICATION.search(title_text) or _CRITERIA_PROPERTY.search(title_text))
    substantive_application = [raw for raw, sentence in sentences
                               if _APPLICATION.search(sentence) and _PERFORMANCE.search(sentence)
                               and (_SELF_REPORT.search(sentence) or _CRITERIA_OWN_RESULT.search(sentence))
                               and not _CRITERIA_BACKGROUND.search(sentence)]
    substantive_property = [raw for raw, sentence in sentences
                            if _CRITERIA_PROPERTY.search(sentence)
                            and _CRITERIA_PROPERTY_RESULT.search(sentence)
                            and (_SELF_REPORT.search(sentence) or _CRITERIA_OWN_RESULT.search(sentence))
                            and not _CRITERIA_BACKGROUND.search(sentence)]
    property_focus = title_property or bool(substantive_application or substantive_property)
    new_structure = bool(_CRITERIA_NEW_STRUCTURES.search(title_text)) or any(
        _CRITERIA_NEW_STRUCTURES.search(_normalize(raw)) for raw in signals["synthesis"])
    structural_design = bool(_CRITERIA_STRUCTURAL_DESIGN.search(title_text)) or any(
        _CRITERIA_STRUCTURAL_DESIGN.search(sentence)
        and (_SELF_REPORT.search(sentence) or _CRITERIA_OWN_RESULT.search(sentence))
        and not _CRITERIA_BACKGROUND.search(sentence) for _, sentence in sentences)
    original_construction = bool(signals["synthesis"]) or bool(
        _CRITERIA_CREATION_TITLE.search(title_text) and (new_structure or structural_design))
    structural_analysis = bool(_STRUCTURAL_DETAILS.search(full_text)
                               or _CRITERIA_STRUCTURAL_DESIGN.search(title_text)
                               or (new_structure and _DIFFRACTION.search(abstract_text)))
    strict_crystal = original_construction and structural_analysis and (new_structure or structural_design)
    method_contribution = title_route or modification_title or bool(signals["abstract_method"])
    current = dict(assignments[-1], criterion=CRITERIA[2])
    if current["category"] != CATEGORIES[1] and property_focus:
        current.update(
            category=CATEGORIES[3],
            reason="A functional/property endpoint is foregrounded in the title or supported by an explicit performance/property result.",
            evidence=_criteria_snippets((["Title: " + str(title)] if title_property else [])
                                        + substantive_application + substantive_property),
            review_needed=missing_text or bool(signals["construction_structure"] or framework_method_focus),
        )
    elif current["category"] == CATEGORIES[2]:
        if strict_crystal:
            current.update(
                reason="Original construction plus new-structure/structural-design evidence satisfies the stricter crystal-engineering definition.",
                evidence=_criteria_snippets(["Title: " + str(title)] + signals["synthesis"] + signals["structure"]),
                review_needed=missing_text,
            )
        elif method_contribution:
            current.update(
                category=CATEGORIES[0],
                reason="An explicit preparative/modification contribution is present, but stricter original crystal-design evidence is incomplete.",
                evidence=_criteria_snippets(["Title: " + str(title)] + signals["method"]),
                review_needed=True,
            )
        else:
            current.update(
                category=UNCLASSIFIED,
                reason="Stricter crystal engineering needs original construction plus new-structure/structural-design evidence; available text does not establish both.",
                evidence=_criteria_snippets(["Title: " + str(title)] + signals["structure"]),
                review_needed=True,
            )
    elif current["category"] == CATEGORIES[0]:
        if method_contribution:
            current.update(
                reason=("Explicit framework preparative/modification method contribution."
                        if framework_method_focus else
                        "Explicit preparative/modification method retained; framework scope requires review."),
                evidence=_criteria_snippets(["Title: " + str(title)] + signals["method"]),
                review_needed=missing_text or not framework_method_focus,
            )
        elif strict_crystal:
            current.update(
                category=CATEGORIES[2],
                reason="Original structural construction is explicit; the text does not establish a distinct synthetic-method contribution.",
                evidence=_criteria_snippets(["Title: " + str(title)] + signals["synthesis"] + signals["structure"]),
                review_needed=missing_text,
            )
        elif title_molecular and not catalyst_application:
            current.update(
                reason="Explicit molecular reaction development retained provisionally; scope beyond framework preparation requires review.",
                evidence="Title: " + str(title), review_needed=True,
            )
        else:
            current.update(
                category=UNCLASSIFIED,
                reason="Available text does not establish a preparative-method, chemical-modification, or original structural-design contribution under the stricter definitions.",
                evidence="Title: " + str(title), review_needed=True,
            )
    assignments.append(current)

    primary_computational_title = bool(_COMPUTATIONAL_TITLE.search(title_text)
                                       and not _CRITERIA_SUPPORTING_TITLE.search(title_text))
    current = dict(assignments[-1], criterion=CRITERIA[3])
    if primary_computational_title or signals["computation"]:
        current.update(
            category=CATEGORIES[1],
            reason="Explicit primary computational method/contribution takes priority, including mixed experimental/computational studies.",
            evidence=_criteria_snippets((["Title: " + str(title)] if primary_computational_title else [])
                                        + signals["computation"]),
            review_needed=missing_text or bool(signals["synthesis"] or property_focus),
        )
    assignments.append(current)
    return assignments

In [ ]:
"""Summarize fixed topic criteria after classification; no ratio-based relabeling."""
import math
from pathlib import Path

import pandas as pd


TOPIC_ORDER = ('Chemical synthesis', 'Theory & modeling', 'Crystal engineering',
               'Functional materials', 'Unclassified')


def yn_ratio(y, n):
    if n:
        return float(y) / n
    return float('inf') if y else float('nan')


def ratio_text(value):
    if pd.isna(value):
        return 'undefined'
    if math.isinf(value):
        return 'infinite (N=0)'
    return '{:.2f}:1'.format(value)


def summarize_criteria(assignments, decisions, criteria_order):
    """One classification per paper per criterion, identical corpus and decisions.

    Thresholds diagnose the resulting classifications. They are not used to
    assign labels, remove papers, select a winner, or update the fixed rules.
    """
    if decisions.paper_id.duplicated().any():
        raise ValueError('Decision input must have one row per paper.')
    if assignments.duplicated(['criterion', 'paper_id']).any():
        raise ValueError('A paper has multiple assignments within a criterion.')
    if set(assignments.criterion) != set(criteria_order):
        raise ValueError('Assignments must contain every declared criterion, and no others.')
    if not set(assignments.category).issubset(set(TOPIC_ORDER)):
        raise ValueError('Unknown topic category.')
    if not set(decisions.triage_decision).issubset({'Y', 'N', 'Unknown', 'Conflict'}):
        raise ValueError('Unsupported decision value; missing decisions must be Unknown.')
    expected_ids = set(decisions.paper_id)
    for criterion in criteria_order:
        if set(assignments.loc[assignments.criterion.eq(criterion), 'paper_id']) != expected_ids:
            raise ValueError('Every criterion must classify the entire same corpus: ' + criterion)
    annotated = assignments.merge(decisions[['paper_id', 'triage_decision']], on='paper_id',
                                  how='left', validate='many_to_one')
    counts, targets = [], []
    for criterion in criteria_order:
        papers = annotated.loc[annotated.criterion.eq(criterion)]
        table = pd.crosstab(papers.category, papers.triage_decision).reindex(
            index=TOPIC_ORDER, columns=['Y', 'N', 'Unknown', 'Conflict'], fill_value=0).astype(int)
        table['Before total'] = table.sum(axis=1)
        table['Y:N ratio'] = [yn_ratio(y, n) for y, n in zip(table.Y, table.N)]
        table['Y:N'] = table['Y:N ratio'].map(ratio_text)
        table['Needs topic review'] = papers.groupby('category').review_needed.sum().reindex(
            TOPIC_ORDER, fill_value=0).astype(int)
        table.index.name = 'Category'
        table.insert(0, 'criterion', criterion)
        counts.append(table.reset_index())
        chem = table.loc['Chemical synthesis']
        crystal = table.loc['Crystal engineering']
        functional = table.loc['Functional materials']
        unclassified = int(table.loc['Unclassified', 'Before total'])
        complete = bool(table[['Unknown', 'Conflict']].to_numpy().sum() == 0)
        chem_pass = bool(chem.Y > 0 and chem.Y >= 4 * chem.N)
        crystal6_pass = bool(crystal.Y > 0 and crystal.Y > 6 * crystal.N)
        crystal7_pass = bool(crystal.Y > 0 and crystal.Y > 7 * crystal.N)
        functional_pass = bool(functional.Y > functional.N)
        total = int(table['Before total'].sum())
        assert total == len(decisions)
        targets.append({
            'criterion': criterion,
            'Chemical Y:N': chem['Y:N'], 'Chemical >=4:1': chem_pass,
            'Crystal Y:N': crystal['Y:N'], 'Crystal >6:1': crystal6_pass,
            'Crystal >7:1': crystal7_pass,
            'Functional Y:N': functional['Y:N'], 'Functional Y>N': functional_pass,
            'All targets (crystal >6)': complete and chem_pass and crystal6_pass and functional_pass,
            'All targets (crystal >7)': complete and chem_pass and crystal7_pass and functional_pass,
            'Total papers': total, 'Unclassified': unclassified,
            'Topic coverage (%)': round(100 * (total - unclassified) / total, 2) if total else float('nan'),
            'Complete decisions': complete,
        })
    return annotated, pd.concat(counts, ignore_index=True), pd.DataFrame(targets)


def plot_criterion_comparison(counts, targets, criteria_order, labels, output_dir):
    """Draw every fixed criterion, including Unclassified, on a common count scale."""
    import matplotlib.pyplot as plt
    from matplotlib.ticker import MaxNLocator
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    colors = {'Y': '#8fbd83', 'N': '#efbd6d'}
    max_count = max(1, int(counts[['Y', 'N']].to_numpy().max()))
    panel_titles = [labels[c] for c in criteria_order]

    def draw(ax, criterion, title):
        table = counts.loc[counts.criterion.eq(criterion)].set_index('Category').reindex(TOPIC_ORDER)
        positions = list(range(len(TOPIC_ORDER)))
        for offset, decision in [(-0.17, 'N'), (0.17, 'Y')]:
            bars = ax.barh([p + offset for p in positions], table[decision], height=0.30,
                           color=colors[decision], label=decision)
            ax.bar_label(bars, labels=['{:,}'.format(int(v)) for v in table[decision]],
                         padding=3, fontsize=8)
        ax.set_yticks(positions)
        ax.set_yticklabels(TOPIC_ORDER, fontsize=9)
        ax.invert_yaxis()
        ax.set_xlim(0, max_count * 1.20)
        ax.xaxis.set_major_locator(MaxNLocator(4, integer=True))
        ax.set_xlabel('Unique papers', fontsize=9)
        ax.set_title(title, loc='left', fontsize=11, weight='bold', pad=12)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.grid(axis='x', alpha=0.17)
        ax.set_axisbelow(True)

    fig, axes = plt.subplots(2, 2, figsize=(14, 10.3))
    for ax, criterion, title in zip(axes.flat, criteria_order, panel_titles):
        draw(ax, criterion, title)
    handles, legend_labels = axes.flat[0].get_legend_handles_labels()
    fig.legend(handles, legend_labels, loc='upper right', ncol=2, frameon=False,
               bbox_to_anchor=(0.985, 0.995))
    fig.suptitle('Literature triage: sensitivity to topic definitions', x=0.02, ha='left',
                 fontsize=16, weight='bold')
    fig.text(0.02, 0.027, 'Same papers and recorded decisions in every panel. Rules use paper content, never Y/N.',
             fontsize=9, color='#52606d')
    fig.text(0.02, 0.008, 'Exploratory classifications; Unclassified is retained. Ratio targets do not establish classification accuracy.',
             fontsize=9, color='#52606d')
    fig.tight_layout(rect=[0, 0.055, 1, 0.945], h_pad=3, w_pad=2.4)
    fig.savefig(output_dir / 'all_criteria_Y_N.png', dpi=200, bbox_inches='tight')
    fig.savefig(output_dir / 'all_criteria_Y_N.pdf', bbox_inches='tight')

    for criterion, title in zip(criteria_order, panel_titles):
        single, ax = plt.subplots(figsize=(10.5, 6.5))
        draw(ax, criterion, title)
        ax.legend(loc='upper right', frameon=False, ncol=2)
        target = targets.set_index('criterion').loc[criterion]
        text = 'Y:N: chemical {} | crystal {} | functional {}'.format(
            target['Chemical Y:N'], target['Crystal Y:N'], target['Functional Y:N'])
        single.text(0.02, 0.06, text, fontsize=9, color='#52606d')
        single.text(0.02, 0.025, 'Recorded Y/N decisions; exploratory topic definition; unresolved topics remain visible.',
                    fontsize=9, color='#52606d')
        single.tight_layout(rect=[0, 0.10, 1, 1])
        single.savefig(output_dir / (criterion + '_Y_N.png'), dpi=200, bbox_inches='tight')
        single.savefig(output_dir / (criterion + '_Y_N.pdf'), bbox_inches='tight')
        plt.close(single)
    return fig


def write_comparison_workbook(tables, path):
    """Streaming export keeps large per-paper comparison tables within memory limits."""
    from openpyxl import Workbook
    from openpyxl.cell import WriteOnlyCell
    from openpyxl.utils import get_column_letter
    workbook = Workbook(write_only=True)
    for name, table in tables.items():
        sheet = workbook.create_sheet(name[:31])
        sheet.freeze_panes = 'A2'
        sheet.auto_filter.ref = 'A1:{}{}'.format(get_column_letter(len(table.columns)), len(table) + 1)
        sheet.append(list(table.columns))
        for values in table.itertuples(index=False, name=None):
            row = []
            for value in values:
                if pd.isna(value):
                    value = None
                elif isinstance(value, float) and math.isinf(value):
                    value = 'infinite' if value > 0 else '-infinite'
                cell = WriteOnlyCell(sheet, value=value)
                if isinstance(value, str):
                    cell.data_type = 's'
                row.append(cell)
            sheet.append(row)
    workbook.save(path)

In [ ]:
"""Notebook section: evaluate all fixed criteria, then join recorded decisions."""
COMPARISON_DIR = OUTPUT_DIR / 'criteria_comparison'
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)
definitions = pd.DataFrame([
    {'criterion': key, 'Label': CRITERION_LABELS[key],
     'Definition': CRITERION_DESCRIPTIONS[key]} for key in CRITERIA
])
display(definitions.style.set_properties(**{'white-space': 'normal'}))

# No decision field is passed to the classifier. Freeze all assignments before
# joining Y/N labels or calculating ratios. Manual corrections apply equally.
assignment_rows = []
for record in full_paper_decisions.to_dict('records'):
    variants = assign_topic_criteria(
        record['Article Title'], record['Abstract'], record.get('Document Type', ''),
        baseline_category=record['category'])
    for variant in variants:
        result = dict(variant)
        result.update({
            'paper_id': record['paper_id'], 'Article Title': record['Article Title'],
            'DOI': record.get('DOI', ''), 'baseline_category': record['category'],
            'manual_reviewed': bool(record.get('manual_reviewed', False)),
            'is_review_article': bool(record.get('is_review_article', False)),
        })
        if result['category'] == record['category']:
            result['review_needed'] = bool(result['review_needed'] or record.get('review_needed', False))
        if result['criterion'] == 'baseline_current':
            result['reason'] = record.get('classification_reason', record.get('primary_topic_reason', result['reason']))
            result['evidence'] = record.get('framework_synthesis_evidence', '') or record.get('evidence', result['evidence'])
        if result['manual_reviewed']:
            result['category'] = record['category']
            result['reason'] = 'Existing manual category correction retained across criteria.'
            result['evidence'] = record.get('review_note', '')
            result['review_needed'] = False
        # The review-paper convention takes precedence over all topic variants.
        if result['is_review_article']:
            result['category'] = 'Theory & modeling'
            result['reason'] = 'Identified review article: Theory & modeling by user convention.'
            result['evidence'] = record.get('review_detection_evidence', '')
        assignment_rows.append(result)
assignments = pd.DataFrame(assignment_rows)
assignments['changed_from_baseline'] = assignments.category.ne(assignments.baseline_category)
comparison_papers, comparison_counts, target_checks = summarize_criteria(
    assignments, full_paper_decisions[['paper_id', 'triage_decision']], CRITERIA)

print('Counts: Before total is the full corpus; Y is retained after triage; N is rejected.')
display(comparison_counts[['criterion', 'Category', 'Before total', 'Y', 'N',
                           'Y:N', 'Needs topic review']])
print('Target checks (True = pass; False = fail). No criterion is selected by these checks.')
display(target_checks)
passes = target_checks['All targets (crystal >6)'].sum()
print('{} of {} definitions meet all requested targets using crystal >6:1.'.format(passes, len(CRITERIA)))
passes7 = target_checks['All targets (crystal >7)'].sum()
print('{} of {} definitions meet all requested targets using crystal >7:1.'.format(passes7, len(CRITERIA)))

comparison_figure = plot_criterion_comparison(
    comparison_counts, target_checks, CRITERIA, CRITERION_LABELS, COMPARISON_DIR)
display(comparison_figure)
plt.close(comparison_figure)

transition_counts = comparison_papers.groupby(
    ['criterion', 'baseline_category', 'category', 'triage_decision']).size().rename('Papers').reset_index()
changes = comparison_papers.loc[comparison_papers.changed_from_baseline].copy()
review_check = comparison_papers.loc[comparison_papers.is_review_article]
assert review_check.category.eq('Theory & modeling').all()
assert len(comparison_papers) == len(full_paper_decisions) * len(CRITERIA)
for criterion in CRITERIA:
    subset = comparison_papers.loc[comparison_papers.criterion.eq(criterion)]
    assert subset.triage_decision.value_counts().to_dict() == full_paper_decisions.triage_decision.value_counts().to_dict()

comparison_tables = {
    'criterion_definitions': definitions, 'counts_by_criterion': comparison_counts,
    'target_checks': target_checks, 'paper_assignments': comparison_papers,
    'category_changes': changes, 'transition_counts': transition_counts,
}
for name, table in comparison_tables.items():
    table.to_csv(COMPARISON_DIR / (name + '.csv'), index=False, encoding='utf-8-sig')
write_comparison_workbook(comparison_tables, COMPARISON_DIR / 'topic_criteria_comparison.xlsx')

# Hash the actual embedded criteria code, so the notebook stays self-contained.
comparison_manifest = {
    'run_time_utc': datetime.now(timezone.utc).isoformat(),
    'baseline_classifier_version': CLASSIFIER_VERSION,
    'inputs': manifest['inputs'], 'criteria': definitions.to_dict('records'),
    'corpus_size_per_criterion': len(full_paper_decisions),
    'recorded_decisions_per_criterion': full_paper_decisions.triage_decision.value_counts().to_dict(),
    'classifier_uses_triage_decisions': False,
    'criterion_selected_by_ratio': None,
    'validation': 'Exploratory rules; no expert-labeled accuracy evaluation',
    'review_policy': 'Every identified review remains Theory & modeling',
    'manual_corrections_retained': int(full_paper_decisions.manual_reviewed.sum()),
    'thresholds': {'chemical': 'Y >= 4*N and Y > 0', 'crystal': 'Y > 6*N or Y > 7*N, and Y > 0',
                   'functional': 'Y > N'},
}
if 'In' in globals():
    comparison_manifest['executed_criteria_code_sha256'] = next((
        hashlib.sha256(cell.encode('utf-8')).hexdigest() for cell in reversed(In)
        if re.search(r'(?m)^def assign_topic_criteria\(', cell)), None)
(COMPARISON_DIR / 'comparison_manifest.json').write_text(
    json.dumps(comparison_manifest, indent=2), encoding='utf-8')
print('Verified the same corpus and decisions for every definition. Saved to:', COMPARISON_DIR.resolve())
print('Individual PNG/PDF plots, per-paper evidence, count tables, and Excel audit are saved.')